In [ ]:
!wget https://cf.10xgenomics.com/samples/xenium/1.0.1/Xenium_FFPE_Human_Breast_Cancer_Rep2/Xenium_FFPE_Human_Breast_Cancer_Rep2_outs.zip

In [ ]:
!wget https://cf.10xgenomics.com/samples/cell-exp/7.0.1/SC3pv3_GEX_Breast_Cancer_DTC_Aggr/SC3pv3_GEX_Breast_Cancer_DTC_Aggr_count_analysis.tar.gz
!wget https://cf.10xgenomics.com/samples/cell-exp/7.0.1/SC3pv3_GEX_Breast_Cancer_DTC_Aggr/SC3pv3_GEX_Breast_Cancer_DTC_Aggr_count_filtered_feature_bc_matrix.h5
!wget https://cf.10xgenomics.com/samples/cell-exp/7.0.1/SC3pv3_GEX_Breast_Cancer_DTC_Aggr/SC3pv3_GEX_Breast_Cancer_DTC_Aggr_count_filtered_feature_bc_matrix.tar.gz
!wget https://cf.10xgenomics.com/samples/cell-exp/7.0.1/SC3pv3_GEX_Breast_Cancer_DTC_Aggr/SC3pv3_GEX_Breast_Cancer_DTC_Aggr_count_summary.json

In [ ]:
!unzip -o /content/Xenium_FFPE_Human_Breast_Cancer_Rep2_outs.zip

In [ ]:
import os

# Specify the directory you want to list
directory_path = '.' # '.' refers to the current directory

# List all entries in the directory
contents = os.listdir(directory_path)

print(f"Contents of '{directory_path}':")
for item in contents:
    print(item)

In [ ]:
import os

# Specify the directory you want to list
directory_path = '/content/outs' # '.' refers to the current directory

# List all entries in the directory
contents = os.listdir(directory_path)

print(f"Contents of '{directory_path}':")
for item in contents:
    print(item)

In [ ]:
!pip install scanpy

In [ ]:
!gunzip /content/outs/transcripts.csv.gz

In [ ]:
import pandas as pd
import scanpy as sc

# 1. Load TRANSCRIPTS with coordinates (use Parquet for speed)
transcripts_df = pd.read_csv('/content/outs/transcripts.csv')
# Or if you only have the CSV:
# transcripts_df = pd.read_csv('transcripts.csv.gz', compression='gzip')

transcripts_df.head()

In [ ]:
print(transcripts_df.shape)
total_transcripts = len(transcripts_df)
unique_genes = transcripts_df['feature_name'].nunique()

print(f"Total number of transcripts: {total_transcripts}")
print(f"Number of unique genes: {unique_genes}")

unique_cell_ids_count = transcripts_df['cell_id'].nunique()
print(f"Total unique cell IDs in transcripts file: {unique_cell_ids_count}")

checking CELL-BY-GENE matrix of spatial data

In [ ]:
# 2. Load CELL-BY-GENE matrix of spatial data(the H5 file is standard)
cell_by_gene_adata = sc.read_10x_h5('/content/outs/cell_feature_matrix.h5')
# The data is now in an AnnData object ready for analysis
print(cell_by_gene_adata)
#output : (cell_index, gene_index) = expression_value

In [ ]:
print(cell_by_gene_adata.X)

In [ ]:
print(cell_by_gene_adata.obs_names)

In [ ]:
print(cell_by_gene_adata.var_names)

sc rna seq ref loading

In [ ]:
import scanpy as sc
# Load the H5 file of the scRNA-seq reference data
adata_ref = sc.read_10x_h5("/content/SC3pv3_GEX_Breast_Cancer_DTC_Aggr_count_filtered_feature_bc_matrix.h5")
# Make variable names unique (important for some 10x data)
adata_ref.var_names_make_unique()

In [ ]:
print(adata_ref.X)

In [ ]:
adata_ref

In [ ]:
print(adata_ref.var_names)

In [ ]:
adata_ref.var.head()

In [ ]:
print(adata_ref.obs_names)

Overlap of genes check with ref scrna

In [ ]:
import scanpy as sc
import numpy as np

# Assuming your objects are named:
# cell_by_gene_adata  -> Your Xenium spatial data (313 genes)
# adata_ref           -> Your Chromium scRNA-seq reference (~20k genes)

# 1. Get the gene sets
spatial_genes = set(cell_by_gene_adata.var_names)
reference_genes = set(adata_ref.var_names)

print(f"Genes in spatial data: {len(spatial_genes)}")
print(f"Genes in reference data: {len(reference_genes)}")

# 2. Find the overlap
overlap_genes = spatial_genes.intersection(reference_genes)
print(f"Overlapping genes: {len(overlap_genes)}")

# 3. Calculate key percentages
pct_of_spatial = len(overlap_genes) / len(spatial_genes) * 100
pct_of_reference = len(overlap_genes) / len(reference_genes) * 100

print(f"\n📊 Overlap Statistics:")
print(f"  - {pct_of_spatial:.1f}% of spatial panel genes are in reference ({len(overlap_genes)}/{len(spatial_genes)})")
print(f"  - {pct_of_reference:.2f}% of reference genes are in spatial panel ({len(overlap_genes)}/{len(reference_genes)})")

# 4. Identify missing genes (CRITICAL!)
missing_genes = spatial_genes - reference_genes
print(f"\n⚠️  Genes in spatial panel but MISSING from reference: {len(missing_genes)}")
if len(missing_genes) > 0:
    print("First 10 missing genes:", list(missing_genes)[:10])

In [ ]:
!tar -xzf /content/SC3pv3_GEX_Breast_Cancer_DTC_Aggr_count_analysis.tar.gz

In [ ]:
import os

# Assuming the tar.gz extracted to a directory named 'analysis'
directory_path = './analysis'

# Check if the directory exists before listing
if os.path.exists(directory_path):
    contents = os.listdir(directory_path)
    print(f"Contents of '{directory_path}':")
    for item in contents:
        print(item)
else:
    print(f"Directory '{directory_path}' not found. Please check the extraction path.")

In [ ]:
import os

# Define the path to the clustering directory
directory_path_clustering = './analysis/clustering'

# List all entries in the directory
contents_clustering = os.listdir(directory_path_clustering)

print(f"Contents of '{directory_path_clustering}':")
for item in contents_clustering:
    print(item)

In [ ]:
import pandas as pd
import os

clustering_directories = [
    'gene_expression_kmeans_9_clusters',
    'gene_expression_graphclust',
    'gene_expression_kmeans_5_clusters'
]

base_path = './analysis/clustering'

for cluster_dir in clustering_directories:
    file_path = os.path.join(base_path, cluster_dir, 'clusters.csv')
    if os.path.exists(file_path):
        print(f"\n--- Contents of {cluster_dir}/clusters.csv ---")
        df = pd.read_csv(file_path)
        print(df.head())
    else:
        print(f"\n--- clusters.csv not found in {cluster_dir} ---")

## Explore analysis subdirectories

### Subtask:
Iterate through 'diffexp', 'tsne', 'clustering', 'umap', and 'pca' directories within 'analysis', and recursively list their contents to identify potential data files.


In [ ]:
import os

# Define the root directory for exploration
root_analysis_dir = './analysis'

# Create an empty list to store the paths of all files found
all_files = []

# Use os.walk() to traverse the directory and its subdirectories
for root, dirs, files in os.walk(root_analysis_dir):
    for file in files:
        # Construct the full file path
        full_file_path = os.path.join(root, file)
        all_files.append(full_file_path)

# Print each file path
print(f"All files within '{root_analysis_dir}':")
for f_path in all_files:
    print(f_path)

In [ ]:
import pandas as pd

# Initialize a dictionary to store headers of CSV files
csv_headers = {}

# Iterate through all identified files
for f_path in all_files:
    # Check if the file is a CSV
    if f_path.endswith('.csv'):
        try:
            # Load only the header (first few rows) to inspect column names
            df_head = pd.read_csv(f_path, nrows=0)
            csv_headers[f_path] = df_head.columns.tolist()
            print(f"\n--- Headers for {f_path} ---")
            print(df_head.columns.tolist())
        except Exception as e:
            print(f"\nError reading {f_path}: {e}")




### Reasoning:
differential_expression.csv — what this file represents

Each row = one gene
Each set of three columns = statistics for one cluster

This file answers:

“For each gene, how strongly is it associated with each cluster?”Each gene is tested against each cluster to see whether it is a marker gene for that cluster.


## Select and load clustering data

### Subtask:
Load the 'clusters.csv' file from the 'gene_expression_graphclust' directory (as suggested for graphclust) into a DataFrame, and then display the first few rows of the DataFrame.


**Reasoning**:
The subtask requires loading the specified 'clusters.csv' file into a pandas DataFrame and displaying its head. I will construct the file path and use `pd.read_csv` and `head()` for this purpose.



In [ ]:
import pandas as pd
import os

# 1. Define the file path for clusters.csv in the gene_expression_graphclust directory
file_path_graphclust = './analysis/clustering/gene_expression_graphclust/clusters.csv'

# 2. Load the CSV file into a pandas DataFrame named clusters_df
clusters_df = pd.read_csv(file_path_graphclust)

# 3. Display the first five rows of clusters_df
print(f"--- Contents of {file_path_graphclust} ---")
clusters_df.head()

## Load differential expression data


In [ ]:
import pandas as pd
import os

# 1. Define the file path for differential_expression.csv in the gene_expression_graphclust directory
differential_expression_filepath = './analysis/diffexp/gene_expression_graphclust/differential_expression.csv'

# 2. Load the CSV file into a pandas DataFrame named diff_exp_df
diff_exp_df = pd.read_csv(differential_expression_filepath)

# 3. Display the first five rows of diff_exp_df
print(f"--- Contents of {differential_expression_filepath} ---")
diff_exp_df.head()

## Identify top marker genes per cluster

### Subtask:
Process the loaded differential expression data (`diff_exp_df`) to extract top marker genes for each cluster, based on high log2 fold change, low adjusted p-value, and decent mean counts.


In [ ]:
import re

# 1. Initialize an empty dictionary to store marker genes
marker_genes_per_cluster = {}

# 2. Determine the number of clusters
# Find all column names that start with 'Cluster' and contain 'Mean Counts'
cluster_mean_counts_cols = [col for col in diff_exp_df.columns if re.match(r'Cluster \d+ Mean Counts', col)]
# Extract cluster numbers and find the maximum to get the total number of clusters
cluster_numbers = sorted([int(re.search(r'Cluster (\d+)', col).group(1)) for col in cluster_mean_counts_cols])
num_clusters = max(cluster_numbers) if cluster_numbers else 0

print(f"Identified {num_clusters} clusters.")

# Define thresholds
log2fc_threshold = 0.5
p_value_threshold = 0.05
mean_counts_threshold = 0.1
top_n_genes = 10

# 3. For each cluster, identify top marker genes
for cluster_num in range(1, num_clusters + 1):
    print(f"\nProcessing Cluster {cluster_num}...")

    # a. Filter diff_exp_df to select relevant columns for the current cluster
    current_cluster_cols = [
        'Feature Name',
        f'Cluster {cluster_num} Mean Counts',
        f'Cluster {cluster_num} Log2 fold change',
        f'Cluster {cluster_num} Adjusted p value'
    ]

    # Ensure all required columns exist for the current cluster
    if not all(col in diff_exp_df.columns for col in current_cluster_cols):
        print(f"Warning: Missing columns for Cluster {cluster_num}. Skipping.")
        continue

    cluster_data = diff_exp_df[current_cluster_cols].copy()

    # b. Rename columns for easier manipulation
    cluster_data.columns = ['Feature Name', 'Mean Counts', 'Log2FC', 'Adjusted p value']

    # c. Filter the data for significantly upregulated genes
    filtered_genes = cluster_data[
        (cluster_data['Log2FC'] > log2fc_threshold) &
        (cluster_data['Adjusted p value'] < p_value_threshold) &
        (cluster_data['Mean Counts'] > mean_counts_threshold)
    ]

    # d. Sort by Log2FC and select the top N genes
    top_genes = filtered_genes.sort_values(by='Log2FC', ascending=False).head(top_n_genes)

    # e. Store these top N 'Feature Name' genes in the marker_genes_per_cluster dictionary
    if not top_genes.empty:
        marker_genes_per_cluster[f'Cluster {cluster_num}'] = top_genes['Feature Name'].tolist()
        print(f"Found {len(top_genes)} marker genes for Cluster {cluster_num}.")
    else:
        marker_genes_per_cluster[f'Cluster {cluster_num}'] = []
        print(f"No significant marker genes found for Cluster {cluster_num} with current thresholds.")

# 4. Print the marker_genes_per_cluster dictionary
print("\n--- Top Marker Genes Per Cluster ---")
for cluster, genes in marker_genes_per_cluster.items():
    print(f"{cluster}: {genes}")



**Reasoning**:
The subtask requires assigning cell types to clusters based on the identified marker genes. Since the agent does not have an internal database of cell type markers, a manual assignment of plausible cell types will be performed for illustrative purposes, emphasizing that this step typically involves biological expertise and external resources. This will create a mapping from cluster IDs to putative cell type annotations.



=========================BREAK==================

In [ ]:
import pandas as pd
import os

excel_file_path = '/content/Cell_Barcode_Type_Matrices.xlsx'
sheet_name = 'Xenium R2 Fig1-5 (supervised)'

if os.path.exists(excel_file_path):
    cell_type_matrices_df = pd.read_excel(excel_file_path, sheet_name=sheet_name)

    cluster_label_column = 'Cluster'

    if cluster_label_column in cell_type_matrices_df.columns:
        unique_cluster_labels = cell_type_matrices_df[cluster_label_column].unique()

        print(f"\n--- Unique cell types in '{sheet_name}' ---")
        for label in unique_cluster_labels:
            print(f"- {label}")
        print(f"\nTotal unique cell types: {len(unique_cluster_labels)}")
    else:
        print(f"Error: Column '{cluster_label_column}' not found in the Excel sheet.")
else:
    print(f"Error: The file '{excel_file_path}' was not found.")

In [ ]:
# ============================================================
# FIX 3 (REVISED): Research-backed cell-type assignment for scRNA-seq reference
# ============================================================
# We use Cell Ranger's graph-based clustering from the SC3pv3 analysis,
# then assign cell-type labels based on top marker genes matched to
# the 20 published Janesick et al. cell types.
#
# NOTE: Clusters 2, 5, 12, 13, 14 are labeled "Unlabeled" because they
# represent quality artifacts (high mitochondrial) or have no distinguishing
# markers. Consider filtering these out during QC in your pipeline.

import pandas as pd

# Step 1: Load the Cell Ranger graph-based clustering for the scRNA-seq reference
clusters_df = pd.read_csv(
    './analysis/clustering/gene_expression_graphclust/clusters.csv'
)
print(f"scRNA-seq reference: {len(clusters_df):,} cells in {clusters_df['Cluster'].nunique()} clusters")

# Step 2: Define the cell-type assignments based on marker gene analysis
#   Each assignment is justified by the marker genes identified in your
#   earlier differential expression analysis (Cell 35 in original notebook).
#   Labels are drawn from the Janesick et al. (2023) taxonomy.
cell_type_assignments = {
    'Cluster 1':  'Invasive_Tumor',         # GPC5, ABCB1 (drug efflux), SLC transporters → resistant epithelial tumor
    'Cluster 2':  'Unlabeled',              # ALL mitochondrial genes (MT-ND6, MT-ND2...) → dying/stressed cells, not a real cell type
    'Cluster 3':  'Prolif_Invasive_Tumor',  # KIF18B, ASPM, HJURP, KIF2C → mitotic/proliferation markers
    'Cluster 4':  'B_Cells',                # IGHD, IGLC1, IGKV1-5 → immunoglobulin genes = B lymphocytes
    'Cluster 5':  'Unlabeled',              # ALL mitochondrial genes again → dying/stressed cells
    'Cluster 6':  'DCIS_1',                 # PSCA, MIEN1 (HER2-adjacent), PFN1 → HER2+ epithelial/DCIS
    'Cluster 7':  'Invasive_Tumor',         # HSPA6, HSPA1A/B, DDIT4 → stress/hypoxia response in tumor cells
    'Cluster 8':  'DCIS_2',                 # lncRNAs (FIRRE, DUBR), LMO3 → molecularly distinct tumor subpopulation
    'Cluster 9':  'CD8+_T_Cells',           # GZMH, PRF1, FASLG, IFNG, ZNF683, CXCR6 → cytotoxic T cells
    'Cluster 10': 'B_Cells',               # JCHAIN, IGHA1, IGHG1/G2, DERL3 → plasma cells (antibody-secreting B lineage)
    'Cluster 11': 'DCIS_1',                # SCGB2A2 (mammaglobin), KRT23, LTF, LCN2 → luminal epithelial/ductal
    'Cluster 12': 'Unlabeled',              # Only SORCS2 → too few markers to assign confidently
    'Cluster 13': 'Unlabeled',              # No marker genes found
    'Cluster 14': 'Unlabeled',              # No marker genes found
    'Cluster 15': 'Endothelial',            # CLDN5, SOX17, TAL1, GPIHBP1, ESM1, CLEC14A → blood vessel endothelium
    'Cluster 16': 'Macrophages_1',          # FCER1A, CD1E, ITGAX, TREM1, LILRB2, FPR3 → myeloid/macrophage/DC
    'Cluster 17': 'Stromal',               # COL1A2, COL3A1, DCN, LUM, SFRP2 → fibroblasts/stroma
}

# Step 3: Map cluster numbers to cell-type names
#   Create a 'Cluster_Str' column like "Cluster 7" to match our dictionary keys
clusters_df['Cluster_Str'] = 'Cluster ' + clusters_df['Cluster'].astype(str)
clusters_df['Assigned Cell Type'] = clusters_df['Cluster_Str'].map(cell_type_assignments)

# Step 4: Handle any clusters not in our mapping (safety check)
unmapped = clusters_df['Assigned Cell Type'].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped} cells have unmapped clusters — setting to 'Unlabeled'")
    clusters_df['Assigned Cell Type'] = clusters_df['Assigned Cell Type'].fillna('Unlabeled')

# Step 5: Print the assignment summary
print(f"\nCell type assignments:")
print(f"{'Cluster':<12} {'Cell Type':<28} {'Cell Count':>10}")
print("-" * 52)
for c in sorted(clusters_df['Cluster'].unique()):
    ct = clusters_df.loc[clusters_df['Cluster'] == c, 'Assigned Cell Type'].iloc[0]
    count = (clusters_df['Cluster'] == c).sum()
    print(f"Cluster {c:<4} {ct:<28} {count:>10,}")

# Step 6: Attach labels to the scRNA-seq reference AnnData object
annotated_ref_adata = adata_ref.copy()

# Set barcode as index for merging
clusters_indexed = clusters_df.set_index('Barcode')

# Join cluster info into AnnData .obs
annotated_ref_adata.obs = annotated_ref_adata.obs.join(
    clusters_indexed[['Cluster', 'Assigned Cell Type']]
)

# Step 7: Verify the annotation
labeled = annotated_ref_adata.obs['Assigned Cell Type'].notna().sum()
total = len(annotated_ref_adata)
print(f"\nAnnotated reference AnnData:")
print(f"  Total cells: {total:,}")
print(f"  Labeled cells: {labeled:,}")
print(f"\nCell type distribution:")
print(annotated_ref_adata.obs['Assigned Cell Type'].value_counts().to_string())

In [ ]:
import numpy as np

# 1. Access the 'Assigned Cell Type' column and get unique cell types
unique_cell_types = annotated_ref_adata.obs['Assigned Cell Type'].unique()

# 2. Count the number of unique cell types
total_unique_cell_types = len(unique_cell_types)

# 3. Print the total count of unique cell types
print(f"Total number of unique cell types: {total_unique_cell_types}")

# 4. Print the list of all unique cell types
print("\nList of unique cell types:")
for cell_type in unique_cell_types:
    print(f"- {cell_type}")

In [ ]:
'''import pandas as pd
import scanpy as sc

# 1. Load TRANSCRIPTS with coordinates (use Parquet for speed)
transcripts_df = pd.read_csv('/content/outs/transcripts.csv')
# Or if you only have the CSV:
# transcripts_df = pd.read_csv('transcripts.csv.gz', compression='gzip')

transcripts_df.head()'''

In [ ]:
'''
print(transcripts_df.shape)
total_transcripts = len(transcripts_df)
unique_genes = transcripts_df['feature_name'].nunique()

print(f"Total number of transcripts: {total_transcripts}")
print(f"Number of unique genes: {unique_genes}")

unique_cell_ids_count = transcripts_df['cell_id'].nunique()
print(f"Total unique cell IDs in transcripts file: {unique_cell_ids_count}")'''

=====================NEW CODE FOR TRANSCRIPTS FILE'S CELL TYPE LABELING=========

In [ ]:
# ============================================================
# FIX 2: Use published expert cell-type annotations for Xenium Rep2
# ============================================================
# Instead of manually guessing cell types from marker genes,
# we use the supervised annotations from the Janesick et al. (2023)
# Nature Communications paper. These were created by transferring
# labels from the scFFPE-seq reference using proper label-transfer
# algorithms — far more reliable than manual assignment.

import pandas as pd

# Step 1: Load the published annotations Excel file
#   Make sure you uploaded Cell_Barcode_Type_Matrices.xlsx to /content/
#   The sheet "Xenium R2 Fig1-5 (supervised)" has your Rep2 annotations
xenium_labels = pd.read_excel(
    '/content/Cell_Barcode_Type_Matrices.xlsx',
    sheet_name='Xenium R2 Fig1-5 (supervised)'
)

# Step 2: Look at what we loaded
#   'Barcode' = Xenium cell ID (integer, matches cell_id in transcripts_df)
#   'Cluster' = cell type name (e.g., "Macrophages_1", "Stromal", "DCIS_1")
print("Published Xenium Rep2 annotations:")
print(f"  Total cells annotated: {len(xenium_labels):,}")
print(f"  Columns: {list(xenium_labels.columns)}")
print(f"\n  Cell types found ({xenium_labels['Cluster'].nunique()}):")
for ct in sorted(xenium_labels['Cluster'].unique()):
    count = (xenium_labels['Cluster'] == ct).sum()
    print(f"    {ct}: {count:,} cells")

# Step 3: Rename columns to be clearer for our pipeline
#   'Barcode' -> keep as is (it's the cell ID)
#   'Cluster' -> rename to 'Assigned_Xenium_Cell_Type' for clarity
xenium_labels = xenium_labels.rename(columns={'Cluster': 'Assigned_Xenium_Cell_Type'})

# Step 4: Merge these labels into the transcripts table
#   Each transcript has a 'cell_id' — we match that to the 'Barcode'
#   in the annotations to give every transcript its cell's type.
#   Transcripts in cells not in the annotation file get NaN.
print(f"\nTranscripts before merge: {len(transcripts_df):,}")

merged_transcripts_df = transcripts_df.merge(
    xenium_labels,                    # the published labels
    left_on='cell_id',                # column in transcripts_df
    right_on='Barcode',               # column in xenium_labels
    how='left'                        # keep ALL transcripts, even unmatched ones
)

# Step 5: Drop the redundant 'Barcode' column (we already have 'cell_id')
merged_transcripts_df = merged_transcripts_df.drop(columns=['Barcode'])

# Step 6: Check the results
matched = merged_transcripts_df['Assigned_Xenium_Cell_Type'].notna().sum()
unmatched = merged_transcripts_df['Assigned_Xenium_Cell_Type'].isna().sum()
print(f"\nTranscripts with cell type assigned:    {matched:,}")
print(f"Transcripts without cell type (background/unlabeled): {unmatched:,}")
print(f"Unique cell types in merged data: {merged_transcripts_df['Assigned_Xenium_Cell_Type'].nunique()}")

# Step 7: Also attach the labels to the Xenium cell-by-gene AnnData
#   This is useful for cell-level analyses (clustering, denoising, etc.)
annotated_spatial_adata = cell_by_gene_adata.copy()

# The AnnData obs_names are strings like "1", "2", "3"...
# The annotation Barcodes are integers like 1, 2, 3...
# We need to match them by converting one to the other's type.
label_map = xenium_labels.set_index(
    xenium_labels['Barcode'].astype(str)  # convert int -> string to match obs_names
)['Assigned_Xenium_Cell_Type']

annotated_spatial_adata.obs['cell_type'] = annotated_spatial_adata.obs_names.map(label_map)

labeled_count = annotated_spatial_adata.obs['cell_type'].notna().sum()
total_count = len(annotated_spatial_adata)
print(f"\nXenium AnnData: {labeled_count:,} / {total_count:,} cells have cell-type labels")
print(f"  ({labeled_count/total_count*100:.1f}% labeled)")

In [ ]:
import anndata as ad

# Display the annotated_spatial_adata object
print("--- Annotated Spatial AnnData Object ---")
print(annotated_spatial_adata)

# Display the first few rows of observation metadata (.obs)
print("\n--- Annotated Spatial AnnData .obs (first 5 rows) ---")
display(annotated_spatial_adata.obs.head())

# Display the first few rows of variable metadata (.var)
print("\n--- Annotated Spatial AnnData .var (first 5 rows) ---")
display(annotated_spatial_adata.var.head())

# Display the shape of the AnnData object
print(f"\n--- Annotated Spatial AnnData Shape ---")
print(f"Number of cells (observations): {annotated_spatial_adata.n_obs}")
print(f"Number of genes (variables): {annotated_spatial_adata.n_vars}")


In [ ]:
# Calculate the number of unique cell IDs
unique_cell_ids = merged_transcripts_df['cell_id'].nunique()

# Calculate the number of unique gene counts
unique_gene_counts = merged_transcripts_df['feature_name'].nunique()

print(f"Number of unique cell IDs in merged transcripts: {unique_cell_ids}")
print(f"Number of unique gene counts in merged transcripts: {unique_gene_counts}")

In [ ]:
display(merged_transcripts_df.head())


======================NEW CODE ENDS, NOTHNG FILTERED SO FAR===================

In [ ]:
# ============================================================
# CLEAN XENIUM PREPROCESSING FOR STEP 4 / STEP 5
# Insert this cell immediately after:
# ======================NEW CODE ENDS===================
#
# Main purpose:
# 1. Use merged_transcripts_df if available, because it already has cell-type labels.
# 2. Filter molecules by:
#       QV >= 20
#       valid cell_id in Xenium cell_feature_matrix.h5
#       gene_id in Xenium 313-gene matrix
#       valid x/y/z coordinates
#       valid Assigned_Xenium_Cell_Type
# 3. Save clean molecules.parquet.
# 4. Rebuild X_raw_counts directly from the cleaned molecule table.
# 5. Update annotated_spatial_adata.X and .layers['raw'] to use clean counts.
#
# Important preserved variable names:
#   merged_transcripts_df
#   molecules
#   filtered_molecules_df
#   shared_genes
#   X_raw_counts
#   X_raw
#   annotated_spatial_adata
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import sparse

print("=" * 80)
print("CLEAN XENIUM PREPROCESSING")
print("=" * 80)

# ------------------------------------------------------------
# 0. Settings
# ------------------------------------------------------------

QV_THRESHOLD = 20
REQUIRE_CELL_TYPE = True

SAVE_DIR = "/content"
MOLECULES_PARQUET_PATH = os.path.join(SAVE_DIR, "molecules.parquet")

# CSV is huge. Keep False unless you really need it.
SAVE_CSV_TOO = False
MOLECULES_CSV_PATH = os.path.join(SAVE_DIR, "molecules.csv")

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

def find_col(df, candidates, required=True):
    """
    Finds the first available column from a list of possible column names.
    """
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find any of these columns: {candidates}")
    return None

# ------------------------------------------------------------
# 2. Choose source transcript table
# ------------------------------------------------------------

# IMPORTANT CORRECTION:
# Use merged_transcripts_df if available because after Cell 46 it already has
# Assigned_Xenium_Cell_Type from the published Xenium cell labels.
if "merged_transcripts_df" in globals():
    source_df = merged_transcripts_df.copy()
    print("Using source_df = merged_transcripts_df")
else:
    source_df = transcripts_df.copy()
    print("Using source_df = transcripts_df")

print(f"Source table rows: {len(source_df):,}")
print(f"Source table columns: {list(source_df.columns)}")

# ------------------------------------------------------------
# 3. Detect important columns
# ------------------------------------------------------------

print("\nDetecting transcript columns...")

gene_col = find_col(
    source_df,
    ["gene_id", "feature_name", "gene", "target", "Gene", "Feature"]
)

cell_col = find_col(
    source_df,
    ["cell_id", "cell", "Cell_ID", "cellid"]
)

x_col = find_col(
    source_df,
    ["x", "x_location", "global_x", "X"]
)

y_col = find_col(
    source_df,
    ["y", "y_location", "global_y", "Y"]
)

z_col = find_col(
    source_df,
    ["z", "z_location", "global_z", "Z"],
    required=False
)

qv_col = find_col(
    source_df,
    ["quality", "qv", "QV", "Quality", "score"],
    required=True
)

transcript_id_col = find_col(
    source_df,
    ["transcript_id", "molecule_id", "id", "ID"],
    required=False
)

overlaps_nucleus_col = find_col(
    source_df,
    ["overlaps_nucleus", "nucleus_overlap", "in_nucleus"],
    required=False
)

cell_type_col_in_source = find_col(
    source_df,
    ["Assigned_Xenium_Cell_Type", "cell_type", "Cell_Type", "celltype", "annotation"],
    required=False
)

print(f"  gene column        : {gene_col}")
print(f"  cell column        : {cell_col}")
print(f"  x column           : {x_col}")
print(f"  y column           : {y_col}")
print(f"  z column           : {z_col}")
print(f"  QV/quality column  : {qv_col}")
print(f"  transcript ID col  : {transcript_id_col}")
print(f"  nucleus column     : {overlaps_nucleus_col}")
print(f"  cell-type column   : {cell_type_col_in_source}")

# ------------------------------------------------------------
# 4. Define valid cells and valid genes from Xenium cell × gene matrix
# ------------------------------------------------------------

print("\nPreparing valid cell/gene sets from cell_by_gene_adata...")

# This is the official Xenium cell x gene matrix.
# Rows = cells, columns = 313 genes.
matrix_cell_ids_str = pd.Index(cell_by_gene_adata.obs_names.astype(str))
matrix_gene_ids = pd.Index(cell_by_gene_adata.var_names.astype(str))

valid_cell_set = set(matrix_cell_ids_str) #cell_id exists in valid_cell_set
valid_gene_set = set(matrix_gene_ids) #gene_id exists in valid_gene_set

# Keep this important variable name for later cells.
shared_genes = list(matrix_gene_ids)

print(f"  Matrix cells: {len(matrix_cell_ids_str):,}")
print(f"  Matrix genes: {len(matrix_gene_ids):,}")

# ------------------------------------------------------------
# 5. Standardize molecule table columns
# ------------------------------------------------------------

print("\nCreating standardized molecule table...")

merged_transcripts_df = pd.DataFrame({
    "gene_id": source_df[gene_col].astype(str),
    "cell_id": source_df[cell_col],
    "x": pd.to_numeric(source_df[x_col], errors="coerce"),
    "y": pd.to_numeric(source_df[y_col], errors="coerce"),
    "quality": pd.to_numeric(source_df[qv_col], errors="coerce"),
})

if z_col is not None:
    merged_transcripts_df["z"] = pd.to_numeric(source_df[z_col], errors="coerce")
else:
    merged_transcripts_df["z"] = 0.0

if transcript_id_col is not None:
    merged_transcripts_df["transcript_id"] = source_df[transcript_id_col].astype(str)
else:
    # Fallback transcript IDs if no transcript_id column exists.
    merged_transcripts_df["transcript_id"] = [f"tx_{i}" for i in range(len(merged_transcripts_df))]

if overlaps_nucleus_col is not None:
    merged_transcripts_df["overlaps_nucleus"] = source_df[overlaps_nucleus_col].astype(bool)
else:
    merged_transcripts_df["overlaps_nucleus"] = False

# Preserve cell-type labels if they already exist in merged_transcripts_df/source_df.
if cell_type_col_in_source is not None:
    merged_transcripts_df["Assigned_Xenium_Cell_Type"] = source_df[cell_type_col_in_source].astype(str)
else:
    merged_transcripts_df["Assigned_Xenium_Cell_Type"] = np.nan

print("\nBefore filtering:")
print(f"  Rows        : {len(merged_transcripts_df):,}")
print(f"  Unique cells: {merged_transcripts_df['cell_id'].nunique():,}")
print(f"  Unique genes: {merged_transcripts_df['gene_id'].nunique():,}")

# ------------------------------------------------------------
# 6. Clean cell_id
# ------------------------------------------------------------

print("\nCleaning cell_id values...")

# Convert cell_id to numeric.
# Non-numeric/unassigned/background cell IDs become NaN and are removed.
merged_transcripts_df["cell_id_numeric"] = pd.to_numeric(
    merged_transcripts_df["cell_id"],
    errors="coerce"
)

before = len(merged_transcripts_df)
merged_transcripts_df = merged_transcripts_df.dropna(subset=["cell_id_numeric"]).copy()
after = len(merged_transcripts_df)

print(f"  Removed non-numeric/unassigned cell_id rows: {before - after:,}")

merged_transcripts_df["cell_id"] = merged_transcripts_df["cell_id_numeric"].astype(int)
merged_transcripts_df["cell_id_str"] = merged_transcripts_df["cell_id"].astype(str)

# ------------------------------------------------------------
# 7. Apply clean filters
# ------------------------------------------------------------

print("\nApplying clean filters...")

n0 = len(merged_transcripts_df)

# A. QV filter
merged_transcripts_df = merged_transcripts_df[
    merged_transcripts_df["quality"] >= QV_THRESHOLD
].copy()
n_qv = len(merged_transcripts_df)

print(f"  After QV >= {QV_THRESHOLD}: {n_qv:,} rows removed={n0 - n_qv:,}")

# B. Valid cell IDs from Xenium cell x gene matrix
merged_transcripts_df = merged_transcripts_df[
    merged_transcripts_df["cell_id_str"].isin(valid_cell_set)
].copy()
n_valid_cells = len(merged_transcripts_df)

print(f"  After valid cell_id filter: {n_valid_cells:,} rows removed={n_qv - n_valid_cells:,}")

# C. Keep only genes from 313-gene Xenium matrix
merged_transcripts_df = merged_transcripts_df[
    merged_transcripts_df["gene_id"].isin(valid_gene_set)
].copy()
n_valid_genes = len(merged_transcripts_df)

print(f"  After 313-gene filter: {n_valid_genes:,} rows removed={n_valid_cells - n_valid_genes:,}")

# D. Valid coordinates
before_coord = len(merged_transcripts_df)
merged_transcripts_df = merged_transcripts_df.dropna(subset=["x", "y", "z"]).copy()
after_coord = len(merged_transcripts_df)

print(f"  After coordinate filter: {after_coord:,} rows removed={before_coord - after_coord:,}")

# E. Valid cell-type annotation
missing_ct_before = merged_transcripts_df["Assigned_Xenium_Cell_Type"].isna().sum()
print(f"  Missing cell-type rows before optional removal: {missing_ct_before:,}")

if REQUIRE_CELL_TYPE:
    before_ct = len(merged_transcripts_df)

    # Also remove string versions of missing labels if any exist
    bad_ct = merged_transcripts_df["Assigned_Xenium_Cell_Type"].astype(str).isin(
        ["nan", "None", "NA", "NaN", ""]
    )

    merged_transcripts_df = merged_transcripts_df[~bad_ct].copy()
    after_ct = len(merged_transcripts_df)

    print(f"  After cell-type filter: {after_ct:,} rows removed={before_ct - after_ct:,}")

# ------------------------------------------------------------
# 8. Final standard table
# ------------------------------------------------------------

final_cols = [
    "transcript_id",
    "cell_id",
    "gene_id",
    "x",
    "y",
    "z",
    "quality",
    "overlaps_nucleus",
    "Assigned_Xenium_Cell_Type",
]

merged_transcripts_df = merged_transcripts_df[final_cols].copy()

# Preserve variable names expected by later cells
molecules = merged_transcripts_df
filtered_molecules_df = merged_transcripts_df

print("\nFinal clean molecule table:")
print(f"  Rows        : {len(merged_transcripts_df):,}")
print(f"  Unique cells: {merged_transcripts_df['cell_id'].nunique():,}")
print(f"  Unique genes: {merged_transcripts_df['gene_id'].nunique():,}")
print(f"  QV < {QV_THRESHOLD}: {(merged_transcripts_df['quality'] < QV_THRESHOLD).sum():,}")
print(f"  Missing cell type: {merged_transcripts_df['Assigned_Xenium_Cell_Type'].isna().sum():,}")

# ------------------------------------------------------------
# 9. Build clean raw cell x gene count matrix from cleaned molecule table
# ------------------------------------------------------------

print("\nBuilding X_raw_counts from clean molecule table...")

# Use the full official cell list as row structure.
# Cells with no cleaned transcripts remain as all-zero rows.
cell_idx_map = {str(c): i for i, c in enumerate(matrix_cell_ids_str)}
gene_idx_map = {g: j for j, g in enumerate(shared_genes)}

counts_df = (
    merged_transcripts_df
    .assign(cell_id_str=merged_transcripts_df["cell_id"].astype(str))
    .groupby(["cell_id_str", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_raw_counts = np.zeros(
    (len(matrix_cell_ids_str), len(shared_genes)),
    dtype=np.float32
)

rows = counts_df["cell_id_str"].map(cell_idx_map)
cols = counts_df["gene_id"].map(gene_idx_map)
ok = rows.notna() & cols.notna()

X_raw_counts[
    rows[ok].astype(int).to_numpy(),
    cols[ok].astype(int).to_numpy()
] = counts_df.loc[ok, "count"].to_numpy(dtype=np.float32)

# Keep familiar variable name for later cells
X_raw = X_raw_counts.copy()

print(f"  X_raw_counts shape: {X_raw_counts.shape}")
print(f"  X_raw_counts sum  : {X_raw_counts.sum():,.0f}")
print(f"  Molecule rows     : {len(merged_transcripts_df):,}")
print(f"  Difference rows - matrix sum: {len(merged_transcripts_df) - int(X_raw_counts.sum()):,}")

zero_cells = int((X_raw_counts.sum(axis=1) == 0).sum())
print(f"  Cells with zero cleaned transcripts: {zero_cells:,}")

# ------------------------------------------------------------
# 10. Update annotated_spatial_adata to use clean raw counts
# ------------------------------------------------------------

print("\nUpdating annotated_spatial_adata...")

# Save original official 10x matrix before replacing it
if "raw_official_10x" not in annotated_spatial_adata.layers:
    annotated_spatial_adata.layers["raw_official_10x"] = ensure_dense(
        annotated_spatial_adata.X
    ).astype(np.float32)

# Make sure order matches the official Xenium matrix.
# This assumes annotated_spatial_adata is already in the same order as cell_by_gene_adata.
# If not, the code below will reorder it.
if not np.array_equal(annotated_spatial_adata.obs_names.astype(str), matrix_cell_ids_str.astype(str)):
    print("  Reordering annotated_spatial_adata to match cell_by_gene_adata.obs_names...")
    annotated_spatial_adata = annotated_spatial_adata[matrix_cell_ids_str.astype(str)].copy()

if not np.array_equal(annotated_spatial_adata.var_names.astype(str), matrix_gene_ids.astype(str)):
    print("  Reordering annotated_spatial_adata variables to match cell_by_gene_adata.var_names...")
    annotated_spatial_adata = annotated_spatial_adata[:, matrix_gene_ids.astype(str)].copy()

# Replace main matrix with clean raw molecule-count matrix
annotated_spatial_adata.X = X_raw_counts.copy()

# Store clean raw counts in layers too
annotated_spatial_adata.layers["raw"] = X_raw_counts.copy()
annotated_spatial_adata.layers["raw_molecule_counts_clean"] = X_raw_counts.copy()

# Add useful metadata
annotated_spatial_adata.obs["clean_total_counts"] = X_raw_counts.sum(axis=1)
annotated_spatial_adata.obs["has_clean_transcripts"] = (
    annotated_spatial_adata.obs["clean_total_counts"] > 0
)

print(f"  annotated_spatial_adata shape: {annotated_spatial_adata.shape}")
print(f"  annotated_spatial_adata.X sum: {annotated_spatial_adata.X.sum():,.0f}")
print(f"  Nonzero-count cells: {int(annotated_spatial_adata.obs['has_clean_transcripts'].sum()):,}")
print(f"  Zero-count cells   : {int((~annotated_spatial_adata.obs['has_clean_transcripts']).sum()):,}")

# ------------------------------------------------------------
# 11. Diagnostic comparison with official 10x matrix
# ------------------------------------------------------------

print("\nDiagnostic: clean molecule-count matrix vs official 10x matrix")

X_official = ensure_dense(annotated_spatial_adata.layers["raw_official_10x"]).astype(np.float32)

if X_official.shape == X_raw_counts.shape:
    diff = X_official - X_raw_counts
    abs_diff = np.abs(diff)

    print(f"  Official 10x matrix sum       : {X_official.sum():,.0f}")
    print(f"  Clean molecule-count sum      : {X_raw_counts.sum():,.0f}")
    print(f"  Total absolute difference     : {abs_diff.sum():,.0f}")
    print(f"  Mismatched cell-gene pairs    : {(abs_diff > 1e-6).sum():,}")
    print(f"  Official > clean count pairs  : {(diff > 1e-6).sum():,}")
    print(f"  Clean > official count pairs  : {(diff < -1e-6).sum():,}")
else:
    print("  WARNING: Shape mismatch.")
    print(f"  Official shape: {X_official.shape}")
    print(f"  Clean shape   : {X_raw_counts.shape}")

# ------------------------------------------------------------
# 12. Save outputs
# ------------------------------------------------------------

print("\nSaving clean preprocessing outputs...")

merged_transcripts_df.to_parquet(MOLECULES_PARQUET_PATH, index=False)
print(f"  Saved clean molecules parquet: {MOLECULES_PARQUET_PATH}")
print(f"  Size: {os.path.getsize(MOLECULES_PARQUET_PATH) / 1e6:.1f} MB")

np.save(os.path.join(SAVE_DIR, "X_raw_counts_clean.npy"), X_raw_counts)
print(f"  Saved: {os.path.join(SAVE_DIR, 'X_raw_counts_clean.npy')}")

if SAVE_CSV_TOO:
    merged_transcripts_df.to_csv(MOLECULES_CSV_PATH, index=False)
    print(f"  Saved clean molecules CSV: {MOLECULES_CSV_PATH}")

print("\n" + "=" * 80)
print("DONE: CLEAN PREPROCESSING COMPLETE")
print("=" * 80)

print("\nImportant variables now available for later cells:")
print("  merged_transcripts_df   -> clean molecule table")
print("  molecules               -> same clean molecule table")
print("  filtered_molecules_df   -> same clean molecule table")
print("  shared_genes            -> 313 matrix genes")
print("  X_raw_counts            -> clean cell x gene molecule-count matrix")
print("  X_raw                   -> same clean raw count matrix")
print("  annotated_spatial_adata -> .X and .layers['raw'] use clean raw counts")

In [ ]:
# ============================================================
# VIEW + SANITY CHECK CLEAN MOLECULE TABLE AND X_raw_counts
# ============================================================

import numpy as np
import pandas as pd
from scipy import sparse

print("=" * 80)
print("VIEW + SANITY CHECK: CLEAN MOLECULE TABLE AND X_raw_counts")
print("=" * 80)

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------
# 1. Basic object checks
# ------------------------------------------------------------

required_vars = [
    "merged_transcripts_df",
    "molecules",
    "filtered_molecules_df",
    "X_raw_counts",
    "X_raw",
    "shared_genes",
    "annotated_spatial_adata",
]

print("\nChecking required variables:")
for v in required_vars:
    if v in globals():
        print(f"  ✓ {v} exists")
    else:
        print(f"  ✗ {v} MISSING")

# ------------------------------------------------------------
# 2. View clean molecule table
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("CLEAN MOLECULE TABLE SUMMARY")
print("-" * 80)

print(f"Rows/molecules: {len(merged_transcripts_df):,}")
print(f"Columns: {list(merged_transcripts_df.columns)}")

print("\nFirst 5 rows:")
display(merged_transcripts_df.head())

print("\nColumn dtypes:")
display(merged_transcripts_df.dtypes.to_frame("dtype"))

print("\nBasic counts:")
print(f"Unique cells: {merged_transcripts_df['cell_id'].nunique():,}")
print(f"Unique genes: {merged_transcripts_df['gene_id'].nunique():,}")
print(f"Unique cell types: {merged_transcripts_df['Assigned_Xenium_Cell_Type'].nunique():,}")
print(f"QV < 20 molecules: {(merged_transcripts_df['quality'] < 20).sum():,}")
print(f"Missing cell types: {merged_transcripts_df['Assigned_Xenium_Cell_Type'].isna().sum():,}")
print(f"Missing x/y/z: {merged_transcripts_df[['x', 'y', 'z']].isna().any(axis=1).sum():,}")

print("\nCell-type molecule counts:")
display(
    merged_transcripts_df["Assigned_Xenium_Cell_Type"]
    .value_counts()
    .rename_axis("cell_type")
    .reset_index(name="n_molecules")
)

print("\nTop 20 genes by molecule count:")
display(
    merged_transcripts_df["gene_id"]
    .value_counts()
    .head(20)
    .rename_axis("gene_id")
    .reset_index(name="n_molecules")
)

# ------------------------------------------------------------
# 3. View X_raw_counts
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("X_raw_counts SUMMARY")
print("-" * 80)

print(f"X_raw_counts shape: {X_raw_counts.shape}")
print(f"X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"X_raw_counts dtype: {X_raw_counts.dtype}")
print(f"Number of genes in shared_genes: {len(shared_genes):,}")

cell_total_counts = X_raw_counts.sum(axis=1)
gene_total_counts = X_raw_counts.sum(axis=0)

print(f"Nonzero cells: {(cell_total_counts > 0).sum():,}")
print(f"Zero-count cells: {(cell_total_counts == 0).sum():,}")
print(f"Nonzero genes: {(gene_total_counts > 0).sum():,}")

print("\nCell total count summary:")
display(pd.Series(cell_total_counts).describe().to_frame("cell_total_counts"))

print("\nGene total count summary:")
display(pd.Series(gene_total_counts, index=shared_genes).describe().to_frame("gene_total_counts"))

print("\nTop 20 genes by X_raw_counts:")
display(
    pd.DataFrame({
        "gene_id": shared_genes,
        "matrix_count": gene_total_counts
    })
    .sort_values("matrix_count", ascending=False)
    .head(20)
)

# ------------------------------------------------------------
# 4. Display a small cell x gene slice
# ------------------------------------------------------------

print("\nFirst 5 cells × first 10 genes from X_raw_counts:")
display(
    pd.DataFrame(
        X_raw_counts[:5, :10],
        index=annotated_spatial_adata.obs_names[:5],
        columns=shared_genes[:10]
    )
)

# ------------------------------------------------------------
# 5. Check molecule table counts match X_raw_counts
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("CHECK: molecule table counts vs X_raw_counts")
print("-" * 80)

# Rebuild count matrix from merged_transcripts_df and compare
cell_ids_matrix = annotated_spatial_adata.obs_names.astype(str)
gene_ids_matrix = list(shared_genes)

cell_idx_map_check = {str(c): i for i, c in enumerate(cell_ids_matrix)}
gene_idx_map_check = {g: j for j, g in enumerate(gene_ids_matrix)}

counts_check = (
    merged_transcripts_df
    .assign(cell_id_str=merged_transcripts_df["cell_id"].astype(str))
    .groupby(["cell_id_str", "gene_id"])
    .size()
    .reset_index(name="count")
)

X_check = np.zeros_like(X_raw_counts, dtype=np.float32)

rr = counts_check["cell_id_str"].map(cell_idx_map_check)
cc = counts_check["gene_id"].map(gene_idx_map_check)
ok = rr.notna() & cc.notna()

X_check[
    rr[ok].astype(int).to_numpy(),
    cc[ok].astype(int).to_numpy()
] = counts_check.loc[ok, "count"].to_numpy(dtype=np.float32)

diff = X_raw_counts - X_check
abs_diff = np.abs(diff)

print(f"Rebuilt count matrix sum: {X_check.sum(dtype=np.float64):,.0f}")
print(f"Existing X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
print(f"Max abs difference: {abs_diff.max():.6f}")
print(f"Mismatched cell-gene pairs: {(abs_diff > 1e-6).sum():,}")

if abs_diff.max() < 1e-6:
    print("VERDICT: X_raw_counts exactly matches counts rebuilt from clean molecule table.")
else:
    print("WARNING: X_raw_counts does NOT perfectly match rebuilt molecule counts.")

# ------------------------------------------------------------
# 6. Check X_raw and annotated_spatial_adata.X consistency
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("CHECK: X_raw, X_raw_counts, annotated_spatial_adata.X consistency")
print("-" * 80)

X_adata = ensure_dense(annotated_spatial_adata.X).astype(np.float32)

print(f"X_raw_counts vs X_raw max diff: {np.abs(X_raw_counts - X_raw).max():.6f}")
print(f"X_raw_counts vs annotated_spatial_adata.X max diff: {np.abs(X_raw_counts - X_adata).max():.6f}")

if "raw" in annotated_spatial_adata.layers:
    X_layer_raw = ensure_dense(annotated_spatial_adata.layers["raw"]).astype(np.float32)
    print(f"X_raw_counts vs annotated_spatial_adata.layers['raw'] max diff: {np.abs(X_raw_counts - X_layer_raw).max():.6f}")

# ------------------------------------------------------------
# 7. Check official 10x matrix if available
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("CHECK: clean matrix vs official 10x matrix")
print("-" * 80)

if "raw_official_10x" in annotated_spatial_adata.layers:
    X_official = ensure_dense(annotated_spatial_adata.layers["raw_official_10x"]).astype(np.float32)
    off_diff = X_official - X_raw_counts
    off_abs_diff = np.abs(off_diff)

    print(f"Official 10x matrix sum: {X_official.sum(dtype=np.float64):,.0f}")
    print(f"Clean X_raw_counts sum: {X_raw_counts.sum(dtype=np.float64):,.0f}")
    print(f"Total absolute difference: {off_abs_diff.sum(dtype=np.float64):,.0f}")
    print(f"Mismatched cell-gene pairs: {(off_abs_diff > 1e-6).sum():,}")
    print(f"Official > clean pairs: {(off_diff > 1e-6).sum():,}")
    print(f"Clean > official pairs: {(off_diff < -1e-6).sum():,}")
else:
    print("No raw_official_10x layer found.")

print("\nDONE.")

In [ ]:
# ============================================================
# CHECK NON-INTEGER VALUES IN X_raw_counts
# ============================================================

import numpy as np

print("=" * 70)
print("CHECK: Does X_raw_counts contain non-integer values?")
print("=" * 70)

# Fractional part: how far each value is from its nearest integer
frac_diff = np.abs(X_raw_counts - np.round(X_raw_counts))

non_integer_mask = frac_diff > 1e-6
n_non_integer = int(non_integer_mask.sum())
max_frac_diff = float(frac_diff.max())

print(f"X_raw_counts shape: {X_raw_counts.shape}")
print(f"X_raw_counts dtype: {X_raw_counts.dtype}")
print(f"Total entries: {X_raw_counts.size:,}")
print(f"Non-integer entries: {n_non_integer:,}")
print(f"Max fractional difference from nearest integer: {max_frac_diff:.10f}")

if n_non_integer == 0:
    print("\nVERDICT: X_raw_counts contains only integer-valued counts.")
else:
    print("\nWARNING: X_raw_counts has non-integer values.")

    # Show first few problematic entries
    bad_positions = np.argwhere(non_integer_mask)
    print("\nFirst 20 non-integer entries:")
    for r, c in bad_positions[:20]:
        print(
            f"  row={r}, col={c}, "
            f"cell_id={annotated_spatial_adata.obs_names[r]}, "
            f"gene={shared_genes[c]}, "
            f"value={X_raw_counts[r, c]}, "
            f"nearest_int={np.round(X_raw_counts[r, c])}, "
            f"frac_diff={frac_diff[r, c]:.10f}"
        )

In [ ]:
# ============================================================
# SAVE IMPORTANT CLEAN PREPROCESSING OUTPUTS TO GOOGLE DRIVE
# ============================================================

import os
import json
import shutil
import numpy as np
import pandas as pd
from scipy import sparse

print("=" * 80)
print("SAVE CLEAN PREPROCESSING OUTPUTS TO DRIVE")
print("=" * 80)

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

# ------------------------------------------------------------
# 1. Mount Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------
# 2. Choose output folder in Drive
# ------------------------------------------------------------

DRIVE_DIR = "/content/drive/MyDrive/diffusion/step4_exports"
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"Drive output folder: {DRIVE_DIR}")

# ------------------------------------------------------------
# 3. Define local and Drive paths
# ------------------------------------------------------------

local_molecules_path = "/content/molecules.parquet"
local_xraw_path = "/content/X_raw_counts_clean.npy"
local_adata_path = "/content/annotated_spatial_adata_clean_raw.h5ad"
local_summary_path = "/content/clean_preprocessing_summary.json"

drive_molecules_path = os.path.join(DRIVE_DIR, "molecules.parquet")
drive_xraw_path = os.path.join(DRIVE_DIR, "X_raw_counts.npy")
drive_adata_path = os.path.join(DRIVE_DIR, "annotated_spatial_adata_clean_raw.h5ad")
drive_summary_path = os.path.join(DRIVE_DIR, "clean_preprocessing_summary.json")

# ------------------------------------------------------------
# 4. Save local files first
# ------------------------------------------------------------

print("\nSaving local files...")

# A. Clean molecule table
merged_transcripts_df.to_parquet(local_molecules_path, index=False)
print(f"  Saved local molecules: {local_molecules_path}")

# B. Clean raw count matrix
np.save(local_xraw_path, X_raw_counts)
print(f"  Saved local X_raw_counts: {local_xraw_path}")

# C. AnnData checkpoint
annotated_spatial_adata.write(local_adata_path)
print(f"  Saved local AnnData: {local_adata_path}")

# D. Summary JSON
cell_total_counts = X_raw_counts.sum(axis=1)
gene_total_counts = X_raw_counts.sum(axis=0)

summary = {
    "description": "Clean Xenium preprocessing output before Step 4 denoising",
    "filters": {
        "QV_THRESHOLD": int(QV_THRESHOLD) if "QV_THRESHOLD" in globals() else None,
        "REQUIRE_CELL_TYPE": bool(REQUIRE_CELL_TYPE) if "REQUIRE_CELL_TYPE" in globals() else None,
        "kept_only_cells_in_cell_feature_matrix": True,
        "kept_only_genes_in_cell_feature_matrix": True,
        "kept_only_valid_coordinates": True,
    },
    "clean_molecule_table": {
        "rows": int(len(merged_transcripts_df)),
        "unique_cells": int(merged_transcripts_df["cell_id"].nunique()),
        "unique_genes": int(merged_transcripts_df["gene_id"].nunique()),
        "qv_below_threshold": int((merged_transcripts_df["quality"] < QV_THRESHOLD).sum()) if "QV_THRESHOLD" in globals() else None,
        "missing_cell_type": int(merged_transcripts_df["Assigned_Xenium_Cell_Type"].isna().sum()),
    },
    "X_raw_counts": {
        "shape": list(X_raw_counts.shape),
        "sum": float(X_raw_counts.sum(dtype=np.float64)),
        "nonzero_cells": int((cell_total_counts > 0).sum()),
        "zero_cells": int((cell_total_counts == 0).sum()),
        "nonzero_genes": int((gene_total_counts > 0).sum()),
    },
    "annotated_spatial_adata": {
        "shape": list(annotated_spatial_adata.shape),
        "obs_names_first_5": list(map(str, annotated_spatial_adata.obs_names[:5])),
        "var_names_first_5": list(map(str, annotated_spatial_adata.var_names[:5])),
        "layers": list(annotated_spatial_adata.layers.keys()),
        "obs_columns": list(annotated_spatial_adata.obs.columns),
    },
}

# Add official 10x comparison if available
if "raw_official_10x" in annotated_spatial_adata.layers:
    X_official = ensure_dense(annotated_spatial_adata.layers["raw_official_10x"]).astype(np.float32)
    abs_diff = np.abs(X_official - X_raw_counts)
    summary["official_10x_comparison"] = {
        "official_sum": float(X_official.sum(dtype=np.float64)),
        "clean_sum": float(X_raw_counts.sum(dtype=np.float64)),
        "total_absolute_difference": float(abs_diff.sum(dtype=np.float64)),
        "mismatched_cell_gene_pairs": int((abs_diff > 1e-6).sum()),
    }

with open(local_summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"  Saved local summary: {local_summary_path}")

# ------------------------------------------------------------
# 5. Copy files to Drive
# ------------------------------------------------------------

print("\nCopying files to Drive...")

files_to_copy = [
    (local_molecules_path, drive_molecules_path),
    (local_xraw_path, drive_xraw_path),
    (local_adata_path, drive_adata_path),
    (local_summary_path, drive_summary_path),
]

for src, dst in files_to_copy:
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  ✓ Copied {os.path.basename(src)} -> {dst}")
        print(f"    Size: {os.path.getsize(dst) / 1e6:.1f} MB")
    else:
        print(f"  ✗ Missing local file: {src}")

# ------------------------------------------------------------
# 6. Final Drive presence check
# ------------------------------------------------------------

print("\nFinal Drive file check:")

for _, dst in files_to_copy:
    if os.path.exists(dst):
        print(f"  ✓ {os.path.basename(dst)} ({os.path.getsize(dst) / 1e6:.1f} MB)")
    else:
        print(f"  ✗ MISSING: {dst}")

print("\n" + "=" * 80)
print("DONE: CLEAN PREPROCESSING OUTPUTS SAVED TO DRIVE")
print("=" * 80)

==========PREPROCESSED AND SAVED IN DRIVE TILL HERE==========

In [ ]:
import pandas as pd
import os

# Define the base directory
outs_dir = '/content/outs'

# Load and display cell_boundaries.csv.gz
cell_boundaries_path = os.path.join(outs_dir, 'cell_boundaries.csv.gz')
print(f"--- Contents of {cell_boundaries_path} ---")
cell_boundaries_df = pd.read_csv(cell_boundaries_path)
display(cell_boundaries_df.head())

In [ ]:
import pandas as pd
import os

# Define the base directory
outs_dir = '/content/outs'

# Load and display nucleus_boundaries.csv.gz
nucleus_boundaries_path = os.path.join(outs_dir, 'nucleus_boundaries.csv.gz')
print(f"--- Contents of {nucleus_boundaries_path} ---")
nucleus_boundaries_df = pd.read_csv(nucleus_boundaries_path)
display(nucleus_boundaries_df.head())

In [ ]:
import pandas as pd
import os

# Define the base directory
outs_dir = '/content/outs'

# Load and display cells.csv.gz
cells_path = os.path.join(outs_dir, 'cells.csv.gz')
print(f"--- Contents of {cells_path} ---")
cells_df = pd.read_csv(cells_path)
display(cells_df.head())

In [ ]:
# ============================
# Build cell_data.npz from Xenium outs
# Inputs:
#   /content/outs/cells.csv.gz
#   /content/outs/cell_boundaries.csv.gz
#   /content/outs/nucleus_boundaries.csv.gz
# Output:
#   /content/cell_data.npz
# ============================

import os
import numpy as np
import pandas as pd

# --------- Paths (edit if needed) ----------
OUTS_DIR = "/content/outs"
CELLS_PATH = os.path.join(OUTS_DIR, "cells.csv.gz")
CELL_BOUNDS_PATH = os.path.join(OUTS_DIR, "cell_boundaries.csv.gz")
NUC_BOUNDS_PATH = os.path.join(OUTS_DIR, "nucleus_boundaries.csv.gz")
OUT_NPZ = "/content/cell_data.npz"

# --------- Utilities ----------
def order_vertices_by_angle(verts: np.ndarray, centroid: np.ndarray) -> np.ndarray:
    """
    Robust ordering when boundary files don't include a vertex index.
    Sort vertices by polar angle around a centroid.

    verts: (N,2) float array
    centroid: (2,) float array
    """
    if verts.shape[0] <= 2:
        return verts
    rel = verts - centroid[None, :]
    angles = np.arctan2(rel[:, 1], rel[:, 0])
    order = np.argsort(angles)
    return verts[order]

def build_centroid_map(cells_df: pd.DataFrame) -> dict:
    """
    Build dict: cell_id -> np.array([cx, cy]).
    """
    if not {"cell_id", "x_centroid", "y_centroid"}.issubset(set(cells_df.columns)):
        raise ValueError(
            "cells.csv.gz must contain columns: cell_id, x_centroid, y_centroid"
        )
    centroid_map = {}
    for _, row in cells_df.iterrows():
        cid = int(row["cell_id"])
        centroid_map[cid] = np.array([float(row["x_centroid"]), float(row["y_centroid"])], dtype=np.float32)
    return centroid_map

def sanity_check_vertices(df: pd.DataFrame, name: str):
    """
    Quick sanity check that each cell has multiple vertices.
    """
    counts = df.groupby("cell_id").size()
    desc = counts.describe()
    print(f"\n[{name}] vertices per cell_id summary:\n{desc}\n")
    if desc["50%"] < 10:
        print(f"WARNING: Median vertices per cell is <10 for {name}. "
              f"Polygons may be too coarse or file may be incomplete.")

def build_cell_data_dict(cells_df, cell_bounds_df, nuc_bounds_df=None, angle_sort=True):
    """
    Returns:
      cell_data dict:
        { cell_id: { 'cell_boundary': (N,2), 'centroid': (2,), 'nuc_boundary': (M,2 optional) } }
    """
    centroid_map = build_centroid_map(cells_df)

    # Validate boundary columns
    for df, fname in [(cell_bounds_df, "cell_boundaries"), (nuc_bounds_df, "nucleus_boundaries")]:
        if df is None:
            continue
        if not {"cell_id", "vertex_x", "vertex_y"}.issubset(set(df.columns)):
            raise ValueError(
                f"{fname} file must contain columns: cell_id, vertex_x, vertex_y. "
                f"Got: {list(df.columns)}"
            )

    # Build cell boundary polygons
    cell_data = {}
    for cid, grp in cell_bounds_df.groupby("cell_id"):
        cid = int(cid)
        verts = grp[["vertex_x", "vertex_y"]].to_numpy(dtype=np.float32)

        centroid = centroid_map.get(cid, verts.mean(axis=0).astype(np.float32))
        if angle_sort:
            verts = order_vertices_by_angle(verts, centroid)

        cell_data[cid] = {
            "cell_boundary": verts,
            "centroid": centroid
        }

    # Add nucleus boundary polygons if provided
    if nuc_bounds_df is not None:
        for cid, grp in nuc_bounds_df.groupby("cell_id"):
            cid = int(cid)
            if cid not in cell_data:
                continue
            nverts = grp[["vertex_x", "vertex_y"]].to_numpy(dtype=np.float32)
            if angle_sort:
                nverts = order_vertices_by_angle(nverts, cell_data[cid]["centroid"])
            cell_data[cid]["nuc_boundary"] = nverts

    return cell_data

def save_cell_data_npz(cell_data: dict, out_npz: str):
    """
    Save cell_data dict into a portable NPZ with packed ragged arrays:
      - cell_ids: (C,)
      - centroids: (C,2)
      - cell_offsets: (C+1,)
      - cell_vertices: (sumN,2)
      - nuc_offsets: (C+1,)
      - nuc_vertices: (sumM,2)
      - nuc_present: (C,)
    """
    cell_ids = np.array(sorted(cell_data.keys()), dtype=np.int64)
    C = len(cell_ids)
    if C == 0:
        raise ValueError("cell_data is empty. Check your boundary files / cell_id matching.")

    centroids = np.stack([cell_data[int(cid)]["centroid"] for cid in cell_ids]).astype(np.float32)

    # Pack cell boundaries
    cell_offsets = [0]
    cell_vertices_all = []
    for cid in cell_ids:
        v = cell_data[int(cid)]["cell_boundary"].astype(np.float32)
        cell_vertices_all.append(v)
        cell_offsets.append(cell_offsets[-1] + v.shape[0])
    cell_vertices_all = np.vstack(cell_vertices_all).astype(np.float32)
    cell_offsets = np.array(cell_offsets, dtype=np.int64)

    # Pack nucleus boundaries (optional per cell)
    nuc_offsets = [0]
    nuc_vertices_all = []
    nuc_present = []
    for cid in cell_ids:
        entry = cell_data[int(cid)]
        if "nuc_boundary" in entry and entry["nuc_boundary"] is not None and len(entry["nuc_boundary"]) > 0:
            nv = entry["nuc_boundary"].astype(np.float32)
            nuc_vertices_all.append(nv)
            nuc_offsets.append(nuc_offsets[-1] + nv.shape[0])
            nuc_present.append(1)
        else:
            nuc_offsets.append(nuc_offsets[-1])
            nuc_present.append(0)

    nuc_vertices_all = (
        np.vstack(nuc_vertices_all).astype(np.float32)
        if len(nuc_vertices_all) else np.zeros((0, 2), dtype=np.float32)
    )
    nuc_offsets = np.array(nuc_offsets, dtype=np.int64)
    nuc_present = np.array(nuc_present, dtype=np.int8)

    np.savez_compressed(
        out_npz,
        cell_ids=cell_ids,
        centroids=centroids,
        cell_offsets=cell_offsets,
        cell_vertices=cell_vertices_all,
        nuc_offsets=nuc_offsets,
        nuc_vertices=nuc_vertices_all,
        nuc_present=nuc_present
    )

def quick_plot_one_cell(cell_data: dict, cell_id: int):
    """
    Optional: visualize one cell boundary to confirm polygon ordering.
    """
    import matplotlib.pyplot as plt

    entry = cell_data[cell_id]
    v = entry["cell_boundary"]
    c = entry["centroid"]

    plt.figure(figsize=(5,5))
    plt.plot(v[:,0], v[:,1], "-", linewidth=1)
    plt.scatter(v[:,0], v[:,1], s=4)
    plt.scatter([c[0]], [c[1]], s=25, marker="x")
    plt.title(f"Cell boundary (cell_id={cell_id})")
    plt.gca().set_aspect("equal", "box")
    plt.show()

# --------- Main ----------
print("Loading:", CELLS_PATH)
cells_df = pd.read_csv(CELLS_PATH)

print("Loading:", CELL_BOUNDS_PATH)
cell_bounds_df = pd.read_csv(CELL_BOUNDS_PATH)

print("Loading:", NUC_BOUNDS_PATH)
nuc_bounds_df = pd.read_csv(NUC_BOUNDS_PATH)

# Sanity checks
sanity_check_vertices(cell_bounds_df, "cell_boundaries")
sanity_check_vertices(nuc_bounds_df, "nucleus_boundaries")

# Build dict (angle_sort=True is safest when no vertex order column exists)
cell_data = build_cell_data_dict(
    cells_df=cells_df,
    cell_bounds_df=cell_bounds_df,
    nuc_bounds_df=nuc_bounds_df,
    angle_sort=True
)

print("Built cell_data for", len(cell_data), "cells.")
some_id = next(iter(cell_data.keys()))
print("Example cell_id:", some_id)
print("Keys:", list(cell_data[some_id].keys()))
print("Cell boundary shape:", cell_data[some_id]["cell_boundary"].shape)
if "nuc_boundary" in cell_data[some_id]:
    print("Nucleus boundary shape:", cell_data[some_id]["nuc_boundary"].shape)

# Optional plot to validate ordering (uncomment if you want)
# quick_plot_one_cell(cell_data, some_id)

# Save NPZ
save_cell_data_npz(cell_data, OUT_NPZ)
print("Saved cell_data.npz to:", OUT_NPZ)

# --------- Optional: Verify NPZ contents ----------
d = np.load(OUT_NPZ, allow_pickle=False)
print("\nNPZ keys:", list(d.keys()))
print("cell_ids:", d["cell_ids"].shape)
print("centroids:", d["centroids"].shape)
print("cell_vertices:", d["cell_vertices"].shape)
print("nuc_vertices:", d["nuc_vertices"].shape)
print("nuc_present:", d["nuc_present"].shape)


In [ ]:
# Print the AnnData object to show its structure, including obs and var
print("--- New Annotated Reference AnnData Object ---")
print(annotated_ref_adata)

# Display the first few rows of the observation metadata to confirm cell type annotation
print("\n--- Head of .obs with Assigned Cell Type ---")
print(annotated_ref_adata.obs.head())

In [ ]:
import pandas as pd

# If you already created molecules.parquet with gene_id:
molecules = pd.read_parquet("/content/molecules.parquet")

# If you are still using Xenium transcripts.parquet directly:
# molecules = pd.read_parquet("/content/outs/transcripts.parquet")
# molecules = molecules.rename(columns={"feature_name": "gene_id"})  # only if needed

spatial_genes = set(molecules["gene_id"].astype(str).unique())
print("Spatial measured genes:", len(spatial_genes))


In [ ]:
import numpy as np

print("Example var_names:", list(annotated_ref_adata.var_names[:10]))
print("var columns:", list(annotated_ref_adata.var.columns))

# optional: inspect gene_ids if present
if "gene_ids" in annotated_ref_adata.var.columns:
    print("Example gene_ids:", list(annotated_ref_adata.var["gene_ids"].astype(str).values[:10]))


In [ ]:
def pick_best_gene_field(adata, spatial_genes):
    candidates = [("var_names", adata.var_names.astype(str))]
    for col in ["gene_name", "gene_names", "gene_symbol", "gene_symbols", "gene_ids"]:
        if col in adata.var.columns:
            candidates.append((col, adata.var[col].astype(str).values))

    best = None
    for name, genes in candidates:
        overlap = len(set(genes).intersection(spatial_genes))
        if best is None or overlap > best[0]:
            best = (overlap, name, genes)

    return best  # (overlap_count, field_name, gene_array)

overlap, field_name, gene_array = pick_best_gene_field(annotated_ref_adata, spatial_genes)
print("Best matching gene field:", field_name, "overlap:", overlap)


In [ ]:
import numpy as np

# Use the chosen gene names for matching
scrna_gene_names_all = np.array(gene_array, dtype=str)

common_genes = sorted(list(set(scrna_gene_names_all).intersection(spatial_genes)))
print("Common genes (scRNA ∩ spatial):", len(common_genes))

if len(common_genes) == 0:
    raise ValueError("No overlapping genes. Your scRNA gene naming and spatial gene naming do not match.")


In [ ]:
import scipy.sparse as sp

# Build index mapping from scRNA genes -> column indices
gene_to_col = {g: i for i, g in enumerate(scrna_gene_names_all)}
cols = np.array([gene_to_col[g] for g in common_genes], dtype=int)

# Extract expression submatrix (raw counts expected)
X = annotated_ref_adata.X
X_sub = X[:, cols]

# Keep as dense numpy array only if you can fit it in memory.
# For 12k cells and maybe 200-500 genes, dense is fine.
if sp.issparse(X_sub):
    X_sub = X_sub.toarray()

print("X_sub shape:", X_sub.shape)


In [ ]:
label_col = "Assigned Cell Type"
if label_col not in annotated_ref_adata.obs.columns:
    raise ValueError(f"Missing '{label_col}' in adata.obs. Available: {list(annotated_ref_adata.obs.columns)}")

cell_type_cat = annotated_ref_adata.obs[label_col].astype("category")
cell_type_names = list(cell_type_cat.cat.categories)
cell_types = cell_type_cat.cat.codes.to_numpy(dtype=np.int64)

print("Number of cell types:", len(cell_type_names))
print("First 10 cell type names:", cell_type_names[:10])
print("cell_types shape:", cell_types.shape)


In [ ]:
scrna_ref = {
    "expression": X_sub.astype(np.float32),   # raw counts (or float counts)
    "cell_types": cell_types,                 # integer labels
    "cell_type_names": cell_type_names,       # label names
    "gene_names": common_genes                # MUST match spatial gene_id set
}

print("scrna_ref ready:")
print(" expression:", scrna_ref["expression"].shape)
print(" cell_types:", scrna_ref["cell_types"].shape)
print(" genes:", len(scrna_ref["gene_names"]))


In [ ]:
import numpy as np

np.savez_compressed(
    "/content/scrna_ref.npz",
    expression=scrna_ref["expression"],
    cell_types=scrna_ref["cell_types"],
    gene_names=np.array(scrna_ref["gene_names"], dtype=object),
    cell_type_names=np.array(scrna_ref["cell_type_names"], dtype=object),
)

print("Saved: /content/scrna_ref.npz")


In [ ]:
import numpy as np
import json
import os

# Define the path to the .npz file
npz_file_path = '/content/cell_data.npz'
json_file_path = '/content/cell_data.json'

# 1. Load the .npz file
try:
    with np.load(npz_file_path, allow_pickle=False) as data:
        # Convert numpy arrays to Python lists for JSON serialization
        json_compatible_data = {}
        for key in data.keys():
            json_compatible_data[key] = data[key].tolist()

    # 2. Save the dictionary to a JSON file
    with open(json_file_path, 'w') as f:
        json.dump(json_compatible_data, f, indent=4)

    print(f"Successfully saved {npz_file_path} to {json_file_path}")

except FileNotFoundError:
    print(f"Error: {npz_file_path} not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import numpy as np
import anndata as ad
import pandas as pd

# Define paths
npz_file_path = '/content/scrna_ref.npz'
h5ad_file_path = '/content/scrna_ref.h5ad'

# 1. Load the .npz file
try:
    with np.load(npz_file_path, allow_pickle=True) as data:
        expression = data['expression']
        cell_types_codes = data['cell_types']
        gene_names = data['gene_names']
        cell_type_names_map = data['cell_type_names']

    print("Successfully loaded data from scrna_ref.npz")

    # 2. Create AnnData object
    # Create obs (observations/cells) DataFrame
    # Map integer codes back to cell type names
    cell_type_series = pd.Categorical.from_codes(cell_types_codes, categories=cell_type_names_map)
    obs_df = pd.DataFrame({'cell_type': cell_type_series}, index=[f'cell_{i}' for i in range(expression.shape[0])])

    # Create var (variables/genes) DataFrame
    var_df = pd.DataFrame(index=gene_names)

    # Create the AnnData object
    adata_scrna_ref = ad.AnnData(X=expression, obs=obs_df, var=var_df)

    print("AnnData object created:")
    print(adata_scrna_ref)

    # 3. Save the AnnData object to .h5ad format
    adata_scrna_ref.write_h5ad(h5ad_file_path)

    print(f"Successfully saved AnnData object to {h5ad_file_path}")

except FileNotFoundError:
    print(f"Error: {npz_file_path} not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import pandas as pd

# Define the path to the molecules.parquet file
molecules_parquet_path = '/content/molecules.parquet'

# Load the parquet file into a DataFrame
molecules_df = pd.read_parquet(molecules_parquet_path)

# Display the first 5 rows of the DataFrame
print(f"--- First 5 rows of {molecules_parquet_path} ---")
display(molecules_df.head())

=============================CELL LEVEL DENOISING==================================

In [ ]:
# ==============================================================================
# CELL 1: Install dependencies
# ==============================================================================
!pip install scanpy scikit-learn tqdm --quiet

In [ ]:
# ==============================================================================
# CELL 2: Verify datasets are ready
# ==============================================================================

print("Checking datasets...")
print(f"\nXenium: {annotated_spatial_adata.n_obs:,} cells × {annotated_spatial_adata.n_vars} genes")
print(f"scRNA-seq: {annotated_ref_adata.n_obs:,} cells × {annotated_ref_adata.n_vars} genes")

shared = set(annotated_spatial_adata.var_names) & set(annotated_ref_adata.var_names)
print(f"Shared genes: {len(shared)}")

In [ ]:
# ==============================================================================
# CELL 3: Define all helper functions and classes
# ==============================================================================

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import nbinom
from scipy.special import gammaln
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree, NearestNeighbors
from tqdm import tqdm

def ensure_dense(X):
    """Convert sparse to dense if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return X

def log_normalize(X):
    """Library size normalization + log transform."""
    if X.max() > 100:
        lib_sizes = X.sum(axis=1, keepdims=True)
        X_norm = X / (lib_sizes + 1e-8) * 1e4
        return np.log1p(X_norm)
    return X

In [ ]:
import numpy as np
from scipy import sparse

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

print("annotated_spatial_adata:", annotated_spatial_adata.shape)
print("X sum:", ensure_dense(annotated_spatial_adata.X).sum(dtype=np.float64))

if "raw" in annotated_spatial_adata.layers:
    print("raw layer sum:", ensure_dense(annotated_spatial_adata.layers["raw"]).sum(dtype=np.float64))

if "raw_molecule_counts_clean" in annotated_spatial_adata.layers:
    print("clean raw layer sum:", ensure_dense(annotated_spatial_adata.layers["raw_molecule_counts_clean"]).sum(dtype=np.float64))

print("X_raw_counts sum:", X_raw_counts.sum(dtype=np.float64))
print("max diff .X vs X_raw_counts:", np.abs(ensure_dense(annotated_spatial_adata.X) - X_raw_counts).max())
print("cell type columns:", list(annotated_spatial_adata.obs.columns))

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import nbinom
from scipy.special import gammaln
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import BallTree, NearestNeighbors
from tqdm import tqdm

def ensure_dense(X):
    """Convert sparse to dense if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return X

def log_normalize(X):
    """Library size normalization + log transform."""
    if X.max() > 100:
        lib_sizes = X.sum(axis=1, keepdims=True)
        X_norm = X / (lib_sizes + 1e-8) * 1e4
        return np.log1p(X_norm)
    return X

# -----------------------------------------------------------------------------
# IMPROVEMENT 1: Adaptive Radius Graph
# -----------------------------------------------------------------------------

class AdaptiveRadiusGraph:
    """
    Adaptive neighbor selection based on local density.

    Dense regions → more neighbors
    Sparse regions → fewer neighbors (avoid noise from distant cells)
    """

    def __init__(self, radius=100.0, k_min=5, k_max=30, adaptive_radius=True):
        self.radius = radius
        self.k_min = k_min
        self.k_max = k_max
        self.adaptive_radius = adaptive_radius
        self.tree_ = None
        self.coords_ = None

    def fit(self, spatial_coords):
        """Build spatial index."""
        self.coords_ = spatial_coords
        self.tree_ = BallTree(spatial_coords)

        if self.adaptive_radius:
            counts = self.tree_.query_radius(spatial_coords, r=self.radius,
                                              count_only=True)
            self.local_density_ = counts / (np.pi * self.radius ** 2)
            self.median_density_ = np.median(self.local_density_)
        return self

    def get_neighbors(self, idx):
        """Get adaptive neighbors for one cell."""
        query_point = self.coords_[idx:idx+1]

        # Adapt radius to local density
        if self.adaptive_radius:
            density_ratio = self.local_density_[idx] / (self.median_density_ + 1e-8)
            adaptive_r = self.radius / np.sqrt(density_ratio + 0.1)
            adaptive_r = np.clip(adaptive_r, self.radius * 0.5, self.radius * 2.0)
        else:
            adaptive_r = self.radius

        # Query neighbors
        indices, distances = self.tree_.query_radius(
            query_point, r=adaptive_r, return_distance=True
        )
        indices = indices[0]
        distances = distances[0]

        # Remove self
        mask = indices != idx
        indices = indices[mask]
        distances = distances[mask]

        # Sort and clip to [k_min, k_max]
        sort_idx = np.argsort(distances)
        indices = indices[sort_idx]
        distances = distances[sort_idx]

        if len(indices) < self.k_min:
            # Fall back to k nearest
            all_dist, all_idx = self.tree_.query(query_point, k=self.k_min + 1)
            indices = all_idx[0, 1:]
            distances = all_dist[0, 1:]
        elif len(indices) > self.k_max:
            indices = indices[:self.k_max]
            distances = distances[:self.k_max]

        # Gaussian weights
        sigma = adaptive_r / 2
        weights = np.exp(-distances ** 2 / (2 * sigma ** 2))
        weights = weights /(weights.sum() + 1e-8)
        return indices, weights


# -----------------------------------------------------------------------------
# IMPROVEMENT 2: Proper NB Likelihood
# -----------------------------------------------------------------------------

def nb_cdf(observed, mu, alpha):
    """
    CDF of Negative Binomial distribution.

    Used to detect dropouts: if P(X ≤ observed) is very low,
    the observation is surprisingly small → likely a dropout.
    """
    mu = np.maximum(mu, 1e-8)
    alpha = np.maximum(alpha, 1e-8)

    r = 1.0 / alpha
    p = r / (r + mu)

    return nbinom.cdf(observed, r, p)


def detect_dropouts_strict(observed, mu_expected, alpha, p_threshold=0.01):
    """
    STRICT dropout detection - only flag values that are CLEARLY dropouts.

    Criteria:
        1. P(X <= observed) < p_threshold (statistically unlikely)
        2. Expected value > 0.5 (gene should be expressed)
        3. Observed < 70% of expected (clearly too low)
    """
    cdf = nb_cdf(observed, mu_expected, alpha)

    is_dropout = (
        (cdf < p_threshold) &           # Statistically unlikely
        (mu_expected > 0.5) &           # Expected is substantial
        (observed < 0.7 * mu_expected)  #the observed value must be less than 70% of expected.
    )


    return is_dropout, cdf

# -----------------------------------------------------------------------------
# IMPROVEMENT 3: Calibrated Uncertainty
# -----------------------------------------------------------------------------

class UncertaintyCalibrator:
    """
    Calibrate uncertainty using held-out values.

    1. Hide some values
    2. Predict them + uncertainty
    3. Check: |error| ~ predicted uncertainty?
    4. Learn scale factor to achieve proper coverage
    """
    """FIXED: Uses binary search to find scale that achieves target coverage.
    The old version always made coverage WORSE."""

    def __init__(self, holdout_fraction=0.1, target_coverage=0.68):
        self.holdout_fraction = holdout_fraction
        self.target_coverage = target_coverage
        self.scale_factor_ = 1.0
        self.calibration_stats_ = {}

    def create_holdout_mask(self, n_cells, n_genes, seed=42):
        """Create random holdout mask."""
        rng = np.random.RandomState(seed)
        return rng.random((n_cells, n_genes)) < self.holdout_fraction

    def calibrate(self, predicted, uncertainty, true_values, mask, was_corrected):
        """
        Calibrate uncertainty using held-out values.

        Goal: 68% of true values should fall within ±1σ of prediction."""
        """Only calibrate on corrected values."""
        calibration_mask = mask & was_corrected

        if calibration_mask.sum() < 100:
            print("  Warning: Too few corrected values for calibration")
            self.scale_factor_ = 1.0
            self.calibration_stats_ = {'scale_factor': 1.0, 'n_corrected_holdout': calibration_mask.sum()}
            return uncertainty

        pred_h  = predicted[calibration_mask]
        true_h  = true_values[calibration_mask]
        unc_h  = uncertainty[calibration_mask]

        valid = unc_h > 1e-8
        pred_v, true_v, unc_v = pred_h[valid], true_h[valid], unc_h[valid]
        errors = np.abs(pred_v - true_v)

        # Before
        z_before = errors / unc_v
        cov_1s_before = (z_before < 1.0).mean()

        # FIXED: Binary search for optimal scale
        def coverage(scale):
            return (errors / (unc_v * scale) < 1.0).mean()

        low, high = 0.1, 10.0
        for _ in range(50):
            mid = (low + high) / 2
            if coverage(mid) < self.target_coverage:
                high = mid
            else:
                low = mid

        self.scale_factor_ = np.clip(mid, 0.1, 5.0)

        # After
        z_after = errors / (unc_v * self.scale_factor_)
        cov_1s_after = (z_after < 1.0).mean()

        self.calibration_stats_ = {
            'n_corrected_holdout': calibration_mask.sum(),
            'scale_factor': self.scale_factor_,
            'coverage_1sigma_before': cov_1s_before,
            'coverage_1sigma_after': cov_1s_after
        }

        return uncertainty * self.scale_factor_

    def print_report(self):
        s = self.calibration_stats_
        print(f"\n  Calibration on {s['n_corrected_holdout']:,} corrected held-out values")
        print(f"  Scale factor: {s['scale_factor']:.4f}")
        if 'coverage_1sigma_before' in s:
            print(f"  Coverage: {s['coverage_1sigma_before']:.1%} → {s['coverage_1sigma_after']:.1%}")


In [ ]:

# ==============================================================================
# CELL * : Improved cell type mapping
# ==============================================================================

def map_cell_types_improved(xenium_types, ref_types, verbose=True):
    """Fuzzy matching for cell types."""

    synonyms = {
        'cd4_t': ['cd4', 't_helper', 'cd4+'],
        'cd8_t': ['cd8', 'cytotoxic', 'cd8+'],
        'b_cell': ['b_lymphocyte', 'bcell'],
        'macrophage': ['macro', 'monocyte'],
        'endothelial': ['endo', 'vascular'],
        'fibroblast': ['fibro', 'stromal', 'stroma'],
        'tumor': ['cancer', 'malignant', 'invasive'],
        'dcis': ['ductal', 'carcinoma_in_situ'],
        'prolif': ['proliferating', 'cycling']
    }

    def normalize(s):
        return s.lower().strip().replace('-', '_').replace(' ', '_').replace('+', '_plus')

    def find_match(xt, ref_types):
        xt_norm = normalize(xt)

        # Exact
        for rt in ref_types:
            if normalize(rt) == xt_norm:
                return rt

        # Substring
        for rt in ref_types:
            rt_norm = normalize(rt)
            if xt_norm in rt_norm or rt_norm in xt_norm:
                return rt

        # Synonym
        for key, syns in synonyms.items():
            if key in xt_norm or any(s in xt_norm for s in syns):
                for rt in ref_types:
                    rt_norm = normalize(rt)
                    if key in rt_norm or any(s in rt_norm for s in syns):
                        return rt

        # Word overlap
        xt_words = set(xt_norm.replace('_', ' ').split())
        for rt in ref_types:
            rt_words = set(normalize(rt).replace('_', ' ').split())
            if xt_words & rt_words:
                return rt

        return None

    mapping = {}
    unmapped = []

    for xt in xenium_types:
        match = find_match(xt, ref_types)
        if match:
            mapping[xt] = match
        else:
            unmapped.append(xt)

    if verbose:
        print(f"\nCell type mapping: {len(mapping)}/{len(xenium_types)}")
        if unmapped:
            print(f"  Unmapped: {unmapped}")

    return mapping, unmapped


In [ ]:

# ==============================================================================
# CELL 4: Reference profile computation
# ==============================================================================

def compute_reference_profiles(ref_adata, ct_col='cell_type'):
    """Compute mean expression per cell type from scRNA-seq."""
    print("Computing reference profiles...")
    X = ensure_dense(ref_adata.X)
    X = log_normalize(X)

    cell_types = ref_adata.obs[ct_col].values
    profiles = {}

    for ct in np.unique(cell_types):
        mask = cell_types == ct #Creates a Boolean mask that selects only cells of the current type.
        profiles[ct] = {
            'mean': X[mask].mean(axis=0),
            'std': X[mask].std(axis=0),
            'n_cells': mask.sum()
        }
        print(f"  {ct}: {mask.sum()} cells")

    return profiles



In [ ]:
# ==============================================================================
# CELL 7: Main denoising function (v4 - SELECTIVE)
# ==============================================================================

def denoise_v4_selective(xenium_adata, ref_adata,
                         spatial_weight=0.5,
                         reference_weight=0.3,
                         spatial_radius=100.0,
                         k_min=5,
                         k_max=30,
                         n_pca=50,
                         dropout_p_threshold=0.1,
                         shrinkage=0.8,
                         calibrate=True,
                         holdout_frac=0.1):
    """
    SELECTIVE denoising - only correct detected dropouts.

    KEY DIFFERENCE from v3:
        v3: Changed ALL values → destroyed biological signal
        v4: Only changes dropout values → preserves biology
    """
    print("=" * 60)
    print("SELECTIVE SPATIAL DENOISING (v4)")
    print("Only correcting detected dropouts")
    print("=" * 60)
    print(f"  dropout_p_threshold: {dropout_p_threshold}")
    print(f"  shrinkage: {shrinkage}")

    # Shared genes
    shared_genes = list(set(xenium_adata.var_names) & set(ref_adata.var_names))
    print(f"  Shared genes: {len(shared_genes)}")

    xenium_sub = xenium_adata[:, shared_genes].copy()
    ref_sub = ref_adata[:, shared_genes].copy()

    # Reference profiles
    ref_ct_col = None
    for col in ['cell_type', 'celltype', 'cluster', 'Assigned Cell Type']:
        if col in ref_sub.obs.columns:
            ref_ct_col = col
            break

    ref_profiles = compute_reference_profiles(ref_sub, ref_ct_col)

    # Cell type mapping
    xenium_ct_col = 'cell_type' if 'cell_type' in xenium_sub.obs else 'Assigned_Xenium_Cell_Type'
    xenium_types = xenium_sub.obs[xenium_ct_col].unique()
    type_mapping, unmapped = map_cell_types_improved(xenium_types, list(ref_profiles.keys()))

    # Get RAW data
    X = ensure_dense(xenium_sub.X).astype(float)
    n_cells, n_genes = X.shape

    # Holdout
    if calibrate:
        calibrator = UncertaintyCalibrator(holdout_frac)
        holdout_mask = calibrator.create_holdout_mask(n_cells, n_genes)
        X_for_neighbors = X.copy()
        X_for_neighbors[holdout_mask] = 0
        print(f"  Held out {holdout_mask.sum():,} values")
    else:
        X_for_neighbors = X
        holdout_mask = None

    # PCA
    print("  Running PCA...")
    pca = PCA(n_components=n_pca)
    expr_features = pca.fit_transform(log_normalize(X_for_neighbors))

    # Spatial
    x_col = 'x_centroid' if 'x_centroid' in xenium_sub.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in xenium_sub.obs else 'y'
    spatial_coords = np.column_stack([xenium_sub.obs[x_col].values, xenium_sub.obs[y_col].values])
    scaler = StandardScaler()
    spatial_norm = scaler.fit_transform(spatial_coords)

    # Hybrid
    scale = np.sqrt(n_pca / 2)
    hybrid = np.hstack([expr_features * (1 - spatial_weight), spatial_norm * spatial_weight * scale])

    # KEY: Start with ORIGINAL data
    X_denoised = X.copy()
    uncertainty = np.zeros_like(X, dtype=float)
    was_corrected = np.zeros_like(X, dtype=bool)

    # Process by cell type
    print("  Detecting and correcting dropouts...")
    cell_types = xenium_sub.obs[xenium_ct_col].values

    for ct in tqdm(np.unique(cell_types), desc="Cell types"):
        ct_mask = cell_types == ct
        ct_idx = np.where(ct_mask)[0]
        n_ct = len(ct_idx)

        if n_ct < 10:
            continue

        has_ref = ct in type_mapping
        if has_ref:
            ref_mean = ref_profiles[type_mapping[ct]]['mean']

        ct_spatial = spatial_coords[ct_mask]
        ct_hybrid = hybrid[ct_mask]
        ct_X = X_for_neighbors[ct_mask]

        graph = AdaptiveRadiusGraph(radius=spatial_radius, k_min=k_min, k_max=k_max)
        graph.fit(ct_spatial)

        knn = NearestNeighbors(n_neighbors=min(k_max, n_ct - 1), metric='cosine')
        knn.fit(ct_hybrid)

        for i in range(n_ct):
            global_idx = ct_idx[i]

            spatial_idx, _ = graph.get_neighbors(i)
            _, expr_idx = knn.kneighbors(ct_hybrid[i:i+1])
            expr_idx = expr_idx[0, 1:]

            all_nbrs = np.unique(np.concatenate([spatial_idx, expr_idx]))
            if len(all_nbrs) == 0:
                continue

            neighbor_expr = ct_X[all_nbrs]

            s_dist = np.linalg.norm(ct_spatial[all_nbrs] - ct_spatial[i], axis=1)
            e_dist = np.linalg.norm(ct_hybrid[all_nbrs] - ct_hybrid[i], axis=1)
            s_dist_n = s_dist / (s_dist.max() + 1e-8)
            e_dist_n = e_dist / (e_dist.max() + 1e-8)
            h_dist = (1 - spatial_weight) * e_dist_n + spatial_weight * s_dist_n

            weights = np.exp(-h_dist ** 2 / 0.5)
            weights = weights / (weights.sum() + 1e-8)

            mu_neighbors = np.average(neighbor_expr, axis=0, weights=weights)
            var_neighbors = np.average((neighbor_expr - mu_neighbors) ** 2, axis=0, weights=weights)

            if has_ref:
                var_norm = var_neighbors / (var_neighbors.max() + 1e-8)
                lam = np.clip(reference_weight * (1 + var_norm), 0, 0.8)
                mu_expected = (1 - lam) * mu_neighbors + lam * ref_mean
            else:
                mu_expected = mu_neighbors

            alpha = np.maximum((var_neighbors - mu_neighbors) / (mu_neighbors ** 2 + 1e-8), 0.01)
            unc = np.sqrt(mu_expected + alpha * mu_expected ** 2)
            uncertainty[global_idx] = unc

            # STRICT dropout detection
            observed = X[global_idx]
            is_dropout, _ = detect_dropouts_strict(observed, mu_expected, alpha, dropout_p_threshold)

            # ONLY correct dropouts!
            if is_dropout.any():
                corrected = shrinkage * mu_expected + (1 - shrinkage) * observed
                X_denoised[global_idx, is_dropout] = corrected[is_dropout]
                was_corrected[global_idx, is_dropout] = True

    n_corrected = was_corrected.sum()
    pct_corrected = 100 * n_corrected / (n_cells * n_genes)
    print(f"\n  Dropouts corrected: {n_corrected:,} ({pct_corrected:.2f}%)")

    # Calibrate
    if calibrate:
        print("  Calibrating...")
        uncertainty = calibrator.calibrate(X_denoised, uncertainty, X, holdout_mask, was_corrected)
        calibrator.print_report()
        cal_stats = calibrator.calibration_stats_
    else:
        cal_stats = None

    # Diagnostics
    cv_raw = np.std(X, axis=0) / (np.mean(X, axis=0) + 1e-8)
    cv_den = np.std(X_denoised, axis=0) / (np.mean(X_denoised, axis=0) + 1e-8)
    cv_change = (cv_den - cv_raw) / (cv_raw + 1e-8) * 100
    print("Mean CV before:", cv_raw.mean())
    print("Mean CV after :", cv_den.mean())
    print(f"\n  CV change: {cv_change.mean():+.2f}%")
    print(f"  (Negative = variability decreased = GOOD)")

    metadata = {
        'was_corrected': was_corrected,
        'n_corrected': n_corrected,
        'pct_corrected': pct_corrected,
        'shared_genes': shared_genes,
        'calibration_stats': cal_stats,
        'cv_raw': cv_raw,
        'cv_denoised': cv_den,
        'cv_change_pct': cv_change,
        'type_mapping': type_mapping
    }

    print("=" * 60)
    return X_denoised, uncertainty, metadata

In [ ]:
'''# ==============================================================================
# ABLATION: Global Smoothing Version (v3-style) - ADD TO END OF CELL 88
# ==============================================================================

def denoise_v3_global(xenium_adata, ref_adata,
                      spatial_weight=0.3,
                      reference_weight=0.3,
                      spatial_radius=100.0,
                      k_min=5,
                      k_max=30,
                      n_pca=50,
                      shrinkage=0.8,
                      calibrate=False,
                      holdout_frac=0.1):
    """
    ABLATION: Global smoothing - changes ALL values (v3-style).

    KEY DIFFERENCE FROM BASELINE (v4):
        v4 (selective): Only changes detected dropout values
        v3 (global):    Changes ALL values to neighbor-weighted estimates

    This should make CV WORSE (increase) - proving selective correction is needed.
    """
    print("=" * 60)
    print("ABLATION: GLOBAL SMOOTHING (v3-style)")
    print("WARNING: Changing ALL values - this should HURT performance")
    print("=" * 60)
    print(f"  shrinkage: {shrinkage}")

    # Shared genes
    shared_genes = list(set(xenium_adata.var_names) & set(ref_adata.var_names))
    print(f"  Shared genes: {len(shared_genes)}")

    xenium_sub = xenium_adata[:, shared_genes].copy()
    ref_sub = ref_adata[:, shared_genes].copy()

    # Reference profiles
    ref_ct_col = None
    for col in ['cell_type', 'celltype', 'cluster', 'Assigned Cell Type']:
        if col in ref_sub.obs.columns:
            ref_ct_col = col
            break

    ref_profiles = compute_reference_profiles(ref_sub, ref_ct_col)

    # Cell type mapping
    xenium_ct_col = 'cell_type' if 'cell_type' in xenium_sub.obs else 'Assigned_Xenium_Cell_Type'
    xenium_types = xenium_sub.obs[xenium_ct_col].unique()
    type_mapping, unmapped = map_cell_types_improved(xenium_types, list(ref_profiles.keys()))

    # Get RAW data
    X = ensure_dense(xenium_sub.X).astype(float)
    n_cells, n_genes = X.shape

    # No holdout needed for global smoothing comparison
    X_for_neighbors = X.copy()

    # PCA
    print("  Running PCA...")
    pca = PCA(n_components=n_pca)
    expr_features = pca.fit_transform(log_normalize(X_for_neighbors))

    # Spatial
    x_col = 'x_centroid' if 'x_centroid' in xenium_sub.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in xenium_sub.obs else 'y'
    spatial_coords = np.column_stack([xenium_sub.obs[x_col].values, xenium_sub.obs[y_col].values])
    scaler = StandardScaler()
    spatial_norm = scaler.fit_transform(spatial_coords)

    # Hybrid
    scale = np.sqrt(n_pca / 2)
    hybrid = np.hstack([expr_features * (1 - spatial_weight), spatial_norm * spatial_weight * scale])

    # Start with ORIGINAL data
    X_denoised = X.copy()
    uncertainty = np.zeros_like(X, dtype=float)
    was_corrected = np.zeros_like(X, dtype=bool)

    # Process by cell type
    print("  Applying GLOBAL smoothing (changing ALL values)...")
    cell_types = xenium_sub.obs[xenium_ct_col].values

    for ct in tqdm(np.unique(cell_types), desc="Cell types"):
        ct_mask = cell_types == ct
        ct_idx = np.where(ct_mask)[0]
        n_ct = len(ct_idx)

        if n_ct < 10:
            continue

        has_ref = ct in type_mapping
        if has_ref:
            ref_mean = ref_profiles[type_mapping[ct]]['mean']

        ct_spatial = spatial_coords[ct_mask]
        ct_hybrid = hybrid[ct_mask]
        ct_X = X_for_neighbors[ct_mask]

        graph = AdaptiveRadiusGraph(radius=spatial_radius, k_min=k_min, k_max=k_max)
        graph.fit(ct_spatial)

        knn = NearestNeighbors(n_neighbors=min(k_max, n_ct - 1), metric='cosine')
        knn.fit(ct_hybrid)

        for i in range(n_ct):
            global_idx = ct_idx[i]

            spatial_idx, _ = graph.get_neighbors(i)
            _, expr_idx = knn.kneighbors(ct_hybrid[i:i+1])
            expr_idx = expr_idx[0, 1:]

            all_nbrs = np.unique(np.concatenate([spatial_idx, expr_idx]))
            if len(all_nbrs) == 0:
                continue

            neighbor_expr = ct_X[all_nbrs]

            s_dist = np.linalg.norm(ct_spatial[all_nbrs] - ct_spatial[i], axis=1)
            e_dist = np.linalg.norm(ct_hybrid[all_nbrs] - ct_hybrid[i], axis=1)
            s_dist_n = s_dist / (s_dist.max() + 1e-8)
            e_dist_n = e_dist / (e_dist.max() + 1e-8)
            h_dist = (1 - spatial_weight) * e_dist_n + spatial_weight * s_dist_n

            weights = np.exp(-h_dist ** 2 / 0.5)
            weights = weights / (weights.sum() + 1e-8)

            mu_neighbors = np.average(neighbor_expr, axis=0, weights=weights)
            var_neighbors = np.average((neighbor_expr - mu_neighbors) ** 2, axis=0, weights=weights)

            if has_ref:
                var_norm = var_neighbors / (var_neighbors.max() + 1e-8)
                lam = np.clip(reference_weight * (1 + var_norm), 0, 0.8)
                mu_expected = (1 - lam) * mu_neighbors + lam * ref_mean
            else:
                mu_expected = mu_neighbors

            alpha = np.maximum((var_neighbors - mu_neighbors) / (mu_neighbors ** 2 + 1e-8), 0.01)
            unc = np.sqrt(mu_expected + alpha * mu_expected ** 2)
            uncertainty[global_idx] = unc

            # ==================================================================
            # KEY DIFFERENCE: GLOBAL SMOOTHING - Change ALL values
            # ==================================================================
            # BASELINE (v4 selective) does this:
            #     observed = X[global_idx]
            #     is_dropout, _ = detect_dropouts_strict(observed, mu_expected, alpha, dropout_p_threshold)
            #     if is_dropout.any():
            #         corrected = shrinkage * mu_expected + (1 - shrinkage) * observed
            #         X_denoised[global_idx, is_dropout] = corrected[is_dropout]  # Only dropouts
            #
            # GLOBAL SMOOTHING (v3) does this:
            observed = X[global_idx]
            corrected = shrinkage * mu_expected + (1 - shrinkage) * observed
            X_denoised[global_idx] = corrected      # ALL values changed!
            was_corrected[global_idx] = True        # Mark ALL as corrected
            # ==================================================================

    n_corrected = was_corrected.sum()
    pct_corrected = 100 * n_corrected / (n_cells * n_genes)
    print(f"\\n  Values changed: {n_corrected:,} ({pct_corrected:.2f}%)")
    print(f"  (Should be ~100% for global smoothing)")

    # Diagnostics
    cv_raw = np.std(X, axis=0) / (np.mean(X, axis=0) + 1e-8)
    cv_den = np.std(X_denoised, axis=0) / (np.mean(X_denoised, axis=0) + 1e-8)
    cv_change = (cv_den - cv_raw) / (cv_raw + 1e-8) * 100

    print(f"\\n  CV change: {cv_change.mean():+.2f}%")
    print(f"  (POSITIVE = variability INCREASED = BAD)")
    print(f"  (This SHOULD be positive, proving global smoothing hurts)")

    metadata = {
        'was_corrected': was_corrected,
        'n_corrected': n_corrected,
        'pct_corrected': pct_corrected,
        'shared_genes': shared_genes,
        'calibration_stats': None,
        'cv_raw': cv_raw,
        'cv_denoised': cv_den,
        'cv_change_pct': cv_change,
        'type_mapping': type_mapping
    }

    print("=" * 60)
    return X_denoised, uncertainty, metadata'''

In [ ]:
'''# ==============================================================================
# ABLATION 4: Fixed-k kNN Version (add this at END of Cell 88)
# ==============================================================================

def denoise_v4_fixed_knn(xenium_adata, ref_adata,
                         spatial_weight=0.3,
                         reference_weight=0.3,
                         k_spatial=15,        # NEW: Fixed k for spatial
                         k_expr=15,           # NEW: Fixed k for expression
                         n_pca=50,
                         dropout_p_threshold=0.1,
                         shrinkage=0.8,
                         calibrate=True,
                         holdout_frac=0.1):
    """
    ABLATION 4: Uses standard fixed-k kNN instead of AdaptiveRadiusGraph.

    This is the baseline approach used by most methods (MAGIC, kNN-smoothing).
    Compares: Your adaptive method vs field standard.

    KEY DIFFERENCE:
        Baseline (v4): AdaptiveRadiusGraph - adapts to local density
        This ablation: Fixed-k kNN - always uses exactly k neighbors
    """
    print("=" * 60)
    print("ABLATION 4: FIXED-K KNN (Standard Baseline)")
    print(f"Using k_spatial={k_spatial}, k_expr={k_expr}")
    print("=" * 60)
    print(f"  dropout_p_threshold: {dropout_p_threshold}")
    print(f"  shrinkage: {shrinkage}")

    # Shared genes
    shared_genes = list(set(xenium_adata.var_names) & set(ref_adata.var_names))
    print(f"  Shared genes: {len(shared_genes)}")

    xenium_sub = xenium_adata[:, shared_genes].copy()
    ref_sub = ref_adata[:, shared_genes].copy()

    # Reference profiles
    ref_ct_col = None
    for col in ['cell_type', 'celltype', 'cluster', 'Assigned Cell Type']:
        if col in ref_sub.obs.columns:
            ref_ct_col = col
            break

    ref_profiles = compute_reference_profiles(ref_sub, ref_ct_col)

    # Cell type mapping
    xenium_ct_col = 'cell_type' if 'cell_type' in xenium_sub.obs else 'Assigned_Xenium_Cell_Type'
    xenium_types = xenium_sub.obs[xenium_ct_col].unique()
    type_mapping, unmapped = map_cell_types_improved(xenium_types, list(ref_profiles.keys()))

    # Get RAW data
    X = ensure_dense(xenium_sub.X).astype(float)
    n_cells, n_genes = X.shape

    # Holdout
    if calibrate:
        calibrator = UncertaintyCalibrator(holdout_frac)
        holdout_mask = calibrator.create_holdout_mask(n_cells, n_genes)
        X_for_neighbors = X.copy()
        X_for_neighbors[holdout_mask] = 0
        print(f"  Held out {holdout_mask.sum():,} values")
    else:
        X_for_neighbors = X
        holdout_mask = None

    # PCA
    print("  Running PCA...")
    pca = PCA(n_components=n_pca)
    expr_features = pca.fit_transform(log_normalize(X_for_neighbors))

    # Spatial
    x_col = 'x_centroid' if 'x_centroid' in xenium_sub.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in xenium_sub.obs else 'y'
    spatial_coords = np.column_stack([xenium_sub.obs[x_col].values, xenium_sub.obs[y_col].values])
    scaler = StandardScaler()
    spatial_norm = scaler.fit_transform(spatial_coords)

    # Hybrid features
    scale = np.sqrt(n_pca / 2)
    hybrid = np.hstack([expr_features * (1 - spatial_weight), spatial_norm * spatial_weight * scale])

    # KEY: Start with ORIGINAL data
    X_denoised = X.copy()
    uncertainty = np.zeros_like(X, dtype=float)
    was_corrected = np.zeros_like(X, dtype=bool)

    # Process by cell type
    print("  Detecting and correcting dropouts...")
    cell_types = xenium_sub.obs[xenium_ct_col].values

    for ct in tqdm(np.unique(cell_types), desc="Cell types"):
        ct_mask = cell_types == ct
        ct_idx = np.where(ct_mask)[0]
        n_ct = len(ct_idx)

        if n_ct < 10:
            continue

        has_ref = ct in type_mapping
        if has_ref:
            ref_mean = ref_profiles[type_mapping[ct]]['mean']

        ct_spatial = spatial_coords[ct_mask]
        ct_hybrid = hybrid[ct_mask]
        ct_X = X_for_neighbors[ct_mask]

        # =======================================================================
        # KEY CHANGE: Use fixed-k kNN instead of AdaptiveRadiusGraph
        # =======================================================================
        # ORIGINAL (adaptive radius):
        # graph = AdaptiveRadiusGraph(radius=spatial_radius, k_min=k_min, k_max=k_max)
        # graph.fit(ct_spatial)

        # ABLATION 4: Fixed-k kNN for spatial neighbors
        spatial_knn = NearestNeighbors(n_neighbors=min(k_spatial + 1, n_ct), metric='euclidean')
        spatial_knn.fit(ct_spatial)
        # =======================================================================

        # Expression kNN (same as before)
        expr_knn = NearestNeighbors(n_neighbors=min(k_expr + 1, n_ct), metric='cosine')
        expr_knn.fit(ct_hybrid)

        for i in range(n_ct):
            global_idx = ct_idx[i]

            # =======================================================================
            # KEY CHANGE: Get fixed-k spatial neighbors instead of adaptive radius
            # =======================================================================
            # ORIGINAL:
            # spatial_idx, _ = graph.get_neighbors(i)

            # ABLATION 4: Fixed-k spatial neighbors
            s_distances, s_indices = spatial_knn.kneighbors(ct_spatial[i:i+1])
            spatial_idx = s_indices[0, 1:]  # Remove self (index 0)
            # =======================================================================

            # Expression neighbors (same as before)
            _, expr_idx = expr_knn.kneighbors(ct_hybrid[i:i+1])
            expr_idx = expr_idx[0, 1:]

            all_nbrs = np.unique(np.concatenate([spatial_idx, expr_idx]))
            if len(all_nbrs) == 0:
                continue

            neighbor_expr = ct_X[all_nbrs]

            s_dist = np.linalg.norm(ct_spatial[all_nbrs] - ct_spatial[i], axis=1)
            e_dist = np.linalg.norm(ct_hybrid[all_nbrs] - ct_hybrid[i], axis=1)
            s_dist_n = s_dist / (s_dist.max() + 1e-8)
            e_dist_n = e_dist / (e_dist.max() + 1e-8)
            h_dist = (1 - spatial_weight) * e_dist_n + spatial_weight * s_dist_n

            weights = np.exp(-h_dist ** 2 / 0.5)
            weights = weights / (weights.sum() + 1e-8)

            mu_neighbors = np.average(neighbor_expr, axis=0, weights=weights)
            var_neighbors = np.average((neighbor_expr - mu_neighbors) ** 2, axis=0, weights=weights)

            if has_ref:
                var_norm = var_neighbors / (var_neighbors.max() + 1e-8)
                lam = np.clip(reference_weight * (1 + var_norm), 0, 0.8)
                mu_expected = (1 - lam) * mu_neighbors + lam * ref_mean
            else:
                mu_expected = mu_neighbors

            alpha = np.maximum((var_neighbors - mu_neighbors) / (mu_neighbors ** 2 + 1e-8), 0.01)
            unc = np.sqrt(mu_expected + alpha * mu_expected ** 2)
            uncertainty[global_idx] = unc

            # STRICT dropout detection (same as before)
            observed = X[global_idx]
            is_dropout, _ = detect_dropouts_strict(observed, mu_expected, alpha, dropout_p_threshold)

            # ONLY correct dropouts! (same as before)
            if is_dropout.any():
                corrected = shrinkage * mu_expected + (1 - shrinkage) * observed
                X_denoised[global_idx, is_dropout] = corrected[is_dropout]
                was_corrected[global_idx, is_dropout] = True

    n_corrected = was_corrected.sum()
    pct_corrected = 100 * n_corrected / (n_cells * n_genes)
    print(f"\\n  Dropouts corrected: {n_corrected:,} ({pct_corrected:.2f}%)")

    # Calibrate
    if calibrate:
        print("  Calibrating...")
        uncertainty = calibrator.calibrate(X_denoised, uncertainty, X, holdout_mask, was_corrected)
        calibrator.print_report()
        cal_stats = calibrator.calibration_stats_
    else:
        cal_stats = None

    # Diagnostics
    cv_raw = np.std(X, axis=0) / (np.mean(X, axis=0) + 1e-8)
    cv_den = np.std(X_denoised, axis=0) / (np.mean(X_denoised, axis=0) + 1e-8)
    cv_change = (cv_den - cv_raw) / (cv_raw + 1e-8) * 100

    print(f"\\n  CV change: {cv_change.mean():+.2f}%")
    print(f"  (Negative = variability decreased = GOOD)")

    metadata = {
        'was_corrected': was_corrected,
        'n_corrected': n_corrected,
        'pct_corrected': pct_corrected,
        'shared_genes': shared_genes,
        'calibration_stats': cal_stats,
        'cv_raw': cv_raw,
        'cv_denoised': cv_den,
        'cv_change_pct': cv_change,
        'type_mapping': type_mapping
    }

    print("=" * 60)
    return X_denoised, uncertainty, metadata'''

In [ ]:
# ==============================================================================
# CELL 8: Run the denoising (ACTUAL DENOISING CELL)
# ==============================================================================

# Ensure centroids exist
if 'x_centroid' not in annotated_spatial_adata.obs.columns:
    cells_df_indexed = cells_df.set_index(cells_df['cell_id'].astype(str))[['x_centroid', 'y_centroid']]
    annotated_spatial_adata.obs = annotated_spatial_adata.obs.join(cells_df_indexed)
    print("Added centroids")

annotated_ref_adata.var_names_make_unique()

# Parameters - more conservative dropout detection
DROPOUT_P = 0.35      # Only flag values with <35% chance of being real
SHRINKAGE = 0.87       # How much to replace (0.8 = mostly neighbor estimate)

# Run
X_denoised, uncertainty, metadata = denoise_v4_selective(
    annotated_spatial_adata,
    annotated_ref_adata,
    spatial_weight=0.5,
    reference_weight=0.7,
    spatial_radius=100.0,
    k_min=5,
    k_max=30,
    dropout_p_threshold=DROPOUT_P,
    shrinkage=SHRINKAGE,
    calibrate=False
)

# Store results
shared_genes = metadata['shared_genes']
denoised_adata = annotated_spatial_adata[:, shared_genes].copy()
denoised_adata.layers['raw'] = ensure_dense(denoised_adata.X)
denoised_adata.X = X_denoised
denoised_adata.layers['uncertainty'] = uncertainty
denoised_adata.obs['was_corrected_count'] = metadata['was_corrected'].sum(axis=1)

print(f"\nStored in denoised_adata")


In [ ]:
"""# ==============================================================================
# CELL 8: Run the denoising (uncomment for ablation 3)
# ==============================================================================

# Ensure centroids exist
if 'x_centroid' not in annotated_spatial_adata.obs.columns:
    cells_df_indexed = cells_df.set_index(cells_df['cell_id'].astype(str))[['x_centroid', 'y_centroid']]
    annotated_spatial_adata.obs = annotated_spatial_adata.obs.join(cells_df_indexed)
    print("Added centroids")

annotated_ref_adata.var_names_make_unique()

# Parameters - more conservative dropout detection
DROPOUT_P = 0.35      # Only flag values with <35% chance of being real
SHRINKAGE = 0.87       # How much to replace (0.8 = mostly neighbor estimate)

# Run
X_denoised, uncertainty, metadata = denoise_v3_global(
    annotated_spatial_adata,
    annotated_ref_adata,
    spatial_weight=0.3,
    reference_weight=0.3,
    spatial_radius=100.0,
    k_min=5,
    k_max=30,                 # Fixed k for expression neighbors (same as k_max)
    shrinkage=SHRINKAGE,
    calibrate=False
)

# Store results
shared_genes = metadata['shared_genes']
denoised_adata = annotated_spatial_adata[:, shared_genes].copy()
denoised_adata.layers['raw'] = ensure_dense(denoised_adata.X)
denoised_adata.X = X_denoised
denoised_adata.layers['uncertainty'] = uncertainty
denoised_adata.obs['was_corrected_count'] = metadata['was_corrected'].sum(axis=1)

print(f"\nStored in denoised_adata")

"""


In [ ]:
# ==============================================================================
# CELL 9: Visualize results
# ==============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. CV comparison
ax = axes[0, 0]
cv_raw = metadata['cv_raw']
cv_den = metadata['cv_denoised']
ax.hist(cv_raw, bins=50, alpha=0.5, label=f"Raw (mean={cv_raw.mean():.2f})", density=True)
ax.hist(cv_den, bins=50, alpha=0.5, label=f"Denoised (mean={cv_den.mean():.2f})", density=True)
ax.set_xlabel('Coefficient of Variation')
ax.set_ylabel('Density')
ax.set_title('Gene CV: Before vs After (selective)')
ax.legend()

# 2. CV scatter
ax = axes[0, 1]
ax.scatter(cv_raw, cv_den, alpha=0.3, s=5)
ax.plot([0, cv_raw.max()], [0, cv_raw.max()], 'r--', label='No change')
ax.set_xlabel('CV (Raw)')
ax.set_ylabel('CV (Denoised)')
ax.set_title('CV per gene')
ax.legend()

# 3. CV change distribution
ax = axes[1, 0]
cv_change = metadata['cv_change_pct']
colors = ['green' if x < 0 else 'red' for x in [cv_change.mean()]]
ax.hist(cv_change, bins=50, edgecolor='white')
ax.axvline(cv_change.mean(), color='red', linestyle='--', label=f'Mean: {cv_change.mean():+.1f}%')
ax.axvline(0, color='black', linestyle='-', alpha=0.5)
ax.set_xlabel('CV Change (%)')
ax.set_ylabel('Genes')
ax.set_title('CV change per gene (negative = improvement)')
ax.legend()

# 4. Corrections per cell
ax = axes[1, 1]
corrections = metadata['was_corrected'].sum(axis=1)
ax.hist(corrections, bins=50, edgecolor='white')
ax.axvline(corrections.mean(), color='red', linestyle='--', label=f'Mean: {corrections.mean():.1f}')
ax.set_xlabel('Dropouts corrected per cell')
ax.set_ylabel('Cells')
ax.set_title(f'Dropout corrections ({metadata["pct_corrected"]:.2f}% of values)')
ax.legend()

plt.tight_layout()
plt.savefig('/content/step4_v4_diagnostics.png', dpi=150)
plt.show()


# ==============================================================================
# CELL 10: Compare specific gene before/after
# ==============================================================================

test_gene = 'CD3D'  # T-cell marker

if test_gene in denoised_adata.var_names:
    gene_idx = denoised_adata.var_names.tolist().index(test_gene)

    raw = denoised_adata.layers['raw'][:, gene_idx]
    den = denoised_adata.X[:, gene_idx]
    corrected_mask = metadata['was_corrected'][:, gene_idx]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Distribution
    ax = axes[0]
    ax.hist(raw, bins=50, alpha=0.5, label='Raw', density=True)
    ax.hist(den, bins=50, alpha=0.5, label='Denoised', density=True)
    ax.set_title(f'{test_gene}: Distribution')
    ax.legend()

    # Scatter (colored by whether corrected)
    ax = axes[1]
    ax.scatter(raw[~corrected_mask], den[~corrected_mask], alpha=0.1, s=1, label='Unchanged', c='gray')
    ax.scatter(raw[corrected_mask], den[corrected_mask], alpha=0.5, s=3, label='Corrected', c='red')
    ax.plot([0, raw.max()], [0, raw.max()], 'b--', alpha=0.5)
    ax.set_xlabel('Raw')
    ax.set_ylabel('Denoised')
    ax.set_title(f'{test_gene}: Raw vs Denoised')
    ax.legend()

    # Stats
    ax = axes[2]
    stats_text = f"""
    {test_gene} Statistics:

    Raw:
      Mean: {raw.mean():.2f}
      Std: {raw.std():.2f}
      Zeros: {(raw == 0).sum():,} ({100*(raw==0).mean():.1f}%)

    Denoised:
      Mean: {den.mean():.2f}
      Std: {den.std():.2f}
      Zeros: {(den == 0).sum():,} ({100*(den==0).mean():.1f}%)

    Corrections:
      Values corrected: {corrected_mask.sum():,}
      ({100*corrected_mask.mean():.2f}% of cells)
    """
    ax.text(0.1, 0.5, stats_text, fontsize=11, family='monospace', va='center')
    ax.axis('off')

    plt.tight_layout()
    plt.savefig('/content/step4_v4_gene_example.png', dpi=150)
    plt.show()


# ==============================================================================
# CELL 11: Save outputs
# ==============================================================================

import os
os.makedirs('/content/step4_outputs', exist_ok=True)

denoised_adata.write('/content/step4_outputs/denoised_adata_v4.h5ad')

np.savez_compressed(
    '/content/step4_outputs/denoised_data_v4.npz',
    X_denoised=denoised_adata.X,
    X_raw=denoised_adata.layers['raw'],
    uncertainty=denoised_adata.layers['uncertainty'],
    was_corrected=metadata['was_corrected'],
    gene_names=shared_genes,
    cv_raw=metadata['cv_raw'],
    cv_denoised=metadata['cv_denoised'],
    cv_change=metadata['cv_change_pct']
)

print("Saved to /content/step4_outputs/")


# ==============================================================================
# CELL 12: Summary
# ==============================================================================

print("=" * 60)
print("STEP 4 COMPLETE (v4 - SELECTIVE)")
print("=" * 60)

cv_change = metadata['cv_change_pct'].mean()
print(f"\nKEY METRICS:")
print(f"  CV change: {cv_change:+.2f}%")
if cv_change < 0:
    print(f"    ✓ Variability DECREASED (GOOD!)")
else:
    print(f"    ✗ Variability increased (investigate)")

print(f"  Values corrected: {metadata['n_corrected']:,} ({metadata['pct_corrected']:.2f}%)")
print(f"  Values unchanged: {metadata['was_corrected'].size - metadata['n_corrected']:,}")

print(f"\nv4 IMPROVEMENTS:")
print(f"  • Only detected dropouts are corrected")
print(f"  • Biological signal is preserved")
print(f"  • True zeros are not touched")
print(f"  • Sparse structure maintained")

print("\n" + "=" * 60)
print("Ready for Step 5: Subcellular Molecule-Level Denoising")
print("=" * 60)


In [ ]:
"""
================================================================================
DOWNSTREAM VALIDATION TASKS FOR CELL-LEVEL DENOISING
================================================================================
These tasks validate that denoising improved the data quality.
================================================================================
"""

# ==============================================================================
# INSTALL / IMPORTS
# ==============================================================================

!pip install -q scanpy leidenalg igraph

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import mannwhitneyu

import scanpy as sc
import anndata as ad

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score
)

# ==============================================================================
# HELPER
# ==============================================================================

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)


def get_cell_type_column(adata):
    """
    Finds the correct cell-type column.
    """
    if "cell_type" in adata.obs.columns:
        return "cell_type"
    elif "Assigned_Xenium_Cell_Type" in adata.obs.columns:
        return "Assigned_Xenium_Cell_Type"
    else:
        raise KeyError(
            "No cell-type column found. Expected either 'cell_type' or "
            "'Assigned_Xenium_Cell_Type' in adata.obs."
        )


# ==============================================================================
# TASK 1: MARKER GENE VALIDATION
# ==============================================================================

MARKERS = {
    'CD3D': ['CD4+_T_Cells', 'CD8+_T_Cells'],      # T-cell marker
    'CD19': ['B_Cells'],                            # B-cell marker
    'CD68': ['Macrophages_1', 'Macrophages_2'],     # Macrophage marker
    'EPCAM': ['Invasive_Tumor', 'DCIS_1', 'DCIS_2', 'Prolif_Invasive_Tumor'],
    'COL1A1': ['Stromal'],                          # Fibroblast/stromal marker
    'PECAM1': ['Endothelial'],                      # Endothelial marker
}


def validate_markers(denoised_adata, markers=MARKERS):
    """
    Compare marker gene expression in expected vs unexpected cell types.

    A good denoising should:
        - maintain high expression in expected cell types
        - not artificially inflate expression in unexpected cell types
    """

    ct_col = get_cell_type_column(denoised_adata)

    results = []

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for i, (gene, expected_types) in enumerate(markers.items()):
        if i >= len(axes):
            break

        ax = axes[i]

        if gene not in denoised_adata.var_names:
            ax.text(0.5, 0.5, f"{gene} not found", ha="center")
            ax.set_title(gene)
            continue

        gene_idx = denoised_adata.var_names.tolist().index(gene)

        raw = ensure_dense(denoised_adata.layers["raw"])[:, gene_idx]
        den = ensure_dense(denoised_adata.X)[:, gene_idx]
        cell_types = denoised_adata.obs[ct_col].astype(str).values

        ct_stats = []

        for ct in np.unique(cell_types):
            mask = cell_types == ct

            if mask.sum() < 10:
                continue

            is_expected = any(
                exp.lower() in ct.lower() or ct.lower() in exp.lower()
                for exp in expected_types
            )

            ct_stats.append({
                "cell_type": ct[:20],
                "expected": is_expected,
                "raw_mean": float(raw[mask].mean()),
                "den_mean": float(den[mask].mean()),
                "raw_nonzero": float((raw[mask] > 0).mean() * 100),
                "den_nonzero": float((den[mask] > 0).mean() * 100),
                "n_cells": int(mask.sum()),
            })

        ct_df = pd.DataFrame(ct_stats).sort_values("den_mean", ascending=False)

        if len(ct_df) == 0:
            ax.text(0.5, 0.5, "No valid cell types", ha="center")
            ax.set_title(gene)
            continue

        x = np.arange(len(ct_df))
        width = 0.35

        ax.barh(
            x - width / 2,
            ct_df["raw_mean"],
            width,
            label="Raw",
            alpha=0.7,
            color="lightblue"
        )

        ax.barh(
            x + width / 2,
            ct_df["den_mean"],
            width,
            label="Denoised",
            alpha=0.7,
            color="coral"
        )

        for j, exp in enumerate(ct_df["expected"]):
            if exp:
                ax.scatter(
                    [ct_df.iloc[j]["den_mean"] + 0.1],
                    [j],
                    marker="*",
                    color="green",
                    s=100,
                    zorder=5
                )

        ax.set_yticks(x)
        ax.set_yticklabels(ct_df["cell_type"])
        ax.set_xlabel("Mean Expression")
        ax.set_title(gene)
        ax.legend(loc="lower right")

        results.append(ct_df)

    plt.tight_layout()
    plt.savefig("/content/marker_validation.png", dpi=150)
    plt.show()

    return results


# ==============================================================================
# TASK 2: CLUSTERING QUALITY — CORRECTED VERSION
# ==============================================================================

def evaluate_clustering_quality(
    denoised_adata,
    raw_matrix=None,
    sample_size=50000,
    random_state=42,
    n_pcs=30,
    leiden_resolution=0.5,
):
    """
    Compare clustering quality on raw vs denoised data using the requested setup.

    Requested setup:
        - Raw matrix: X_raw_counts
        - Sample size: 50,000 cells
        - Sampling seed: random_state=42
        - Normalization: Scanpy normalize_total + log1p
        - PCA: Scanpy PCA, 30 PCs
        - Clustering: Leiden, resolution=0.5
        - ARI/NMI: true labels vs Leiden clusters
        - Silhouette: PCA coordinates vs true cell-type labels
        - n_clusters: number of Leiden clusters found by Leiden
    """

    print("\n" + "=" * 70)
    print("TASK 2: CLUSTERING QUALITY — SCANPY / LEIDEN VALIDATION")
    print("=" * 70)

    ct_col = get_cell_type_column(denoised_adata)

    # --------------------------------------------------------------------------
    # 1. Use X_raw_counts as raw matrix
    # --------------------------------------------------------------------------
    if raw_matrix is None:
        if "X_raw_counts" in globals():
            raw_matrix = X_raw_counts
            print("Using global X_raw_counts as raw matrix.")
        else:
            raise ValueError(
                "raw_matrix was not provided and global X_raw_counts was not found."
            )
    else:
        print("Using provided raw_matrix.")

    X_raw_full = ensure_dense(raw_matrix).astype(np.float32)
    X_den_full = ensure_dense(denoised_adata.X).astype(np.float32)

    if X_raw_full.shape != X_den_full.shape:
        raise ValueError(
            f"Shape mismatch: raw matrix shape {X_raw_full.shape}, "
            f"denoised matrix shape {X_den_full.shape}"
        )

    print(f"Raw matrix shape: {X_raw_full.shape}")
    print(f"Denoised matrix shape: {X_den_full.shape}")
    print(f"Raw matrix sum: {X_raw_full.sum(dtype=np.float64):,.0f}")
    print(f"Denoised matrix sum: {X_den_full.sum(dtype=np.float64):,.2f}")
    print(f"Cell-type column: {ct_col}")

    # --------------------------------------------------------------------------
    # 2. Select same 50,000 cells for raw and denoised
    # --------------------------------------------------------------------------
    n_cells = X_raw_full.shape[0]
    n_sample = min(sample_size, n_cells)

    rng = np.random.default_rng(random_state)
    sample_idx = rng.choice(n_cells, size=n_sample, replace=False)

    print(f"Sampled cells: {n_sample:,} / {n_cells:,}")
    print(f"Sampling seed: {random_state}")

    obs_sub = denoised_adata.obs.iloc[sample_idx].copy()
    var_sub = denoised_adata.var.copy()

    true_labels = obs_sub[ct_col].astype(str).values

    # --------------------------------------------------------------------------
    # 3. Create raw and denoised AnnData objects
    # --------------------------------------------------------------------------
    adata_raw = ad.AnnData(
        X=X_raw_full[sample_idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy()
    )

    adata_den = ad.AnnData(
        X=X_den_full[sample_idx, :].copy(),
        obs=obs_sub.copy(),
        var=var_sub.copy()
    )

    # Make variable names unique just in case
    adata_raw.var_names_make_unique()
    adata_den.var_names_make_unique()

    # --------------------------------------------------------------------------
    # 4. Normalize with Scanpy normalize_total + log1p
    # --------------------------------------------------------------------------
    print("\nNormalizing with Scanpy normalize_total + log1p...")

    sc.pp.normalize_total(adata_raw, target_sum=1e4)
    sc.pp.log1p(adata_raw)

    sc.pp.normalize_total(adata_den, target_sum=1e4)
    sc.pp.log1p(adata_den)

    # --------------------------------------------------------------------------
    # 5. PCA using Scanpy, 30 PCs
    # --------------------------------------------------------------------------
    n_pcs_use = min(n_pcs, adata_raw.shape[1] - 1)

    print(f"Running Scanpy PCA with {n_pcs_use} PCs...")

    sc.pp.pca(adata_raw, n_comps=n_pcs_use, random_state=random_state)
    sc.pp.pca(adata_den, n_comps=n_pcs_use, random_state=random_state)

    # --------------------------------------------------------------------------
    # 6. Leiden clustering, resolution=0.5
    # --------------------------------------------------------------------------
    print(f"Running Leiden clustering with resolution={leiden_resolution}...")

    sc.pp.neighbors(adata_raw, n_pcs=n_pcs_use)
    sc.tl.leiden(
        adata_raw,
        resolution=leiden_resolution,
        random_state=random_state,
        key_added="leiden"
    )

    sc.pp.neighbors(adata_den, n_pcs=n_pcs_use)
    sc.tl.leiden(
        adata_den,
        resolution=leiden_resolution,
        random_state=random_state,
        key_added="leiden"
    )

    clusters_raw = adata_raw.obs["leiden"].astype(str).values
    clusters_den = adata_den.obs["leiden"].astype(str).values

    # --------------------------------------------------------------------------
    # 7. ARI/NMI: true labels vs Leiden clusters
    # --------------------------------------------------------------------------
    ari_raw = adjusted_rand_score(true_labels, clusters_raw)
    ari_den = adjusted_rand_score(true_labels, clusters_den)

    nmi_raw = normalized_mutual_info_score(true_labels, clusters_raw)
    nmi_den = normalized_mutual_info_score(true_labels, clusters_den)

    # --------------------------------------------------------------------------
    # 8. Silhouette: PCA coordinates vs true cell-type labels
    # --------------------------------------------------------------------------
    print("Computing silhouette using PCA coordinates vs true cell-type labels...")

    # This computes silhouette over the 50,000 sampled cells.
    # It can be slower than ARI/NMI.
    sil_raw = silhouette_score(
        adata_raw.obsm["X_pca"][:, :n_pcs_use],
        true_labels,
        metric="euclidean"
    )

    sil_den = silhouette_score(
        adata_den.obsm["X_pca"][:, :n_pcs_use],
        true_labels,
        metric="euclidean"
    )

    # --------------------------------------------------------------------------
    # 9. Number of Leiden clusters
    # --------------------------------------------------------------------------
    n_clusters_raw = pd.Series(clusters_raw).nunique()
    n_clusters_den = pd.Series(clusters_den).nunique()

    results = {
        "Raw observed counts": {
            "ARI": ari_raw,
            "NMI": nmi_raw,
            "silhouette": sil_raw,
            "n_clusters": int(n_clusters_raw),
        },
        "Step4 denoised": {
            "ARI": ari_den,
            "NMI": nmi_den,
            "silhouette": sil_den,
            "n_clusters": int(n_clusters_den),
        },
    }

    results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "dataset"})

    # --------------------------------------------------------------------------
    # 10. Print result table
    # --------------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("CLUSTERING QUALITY RESULTS")
    print("=" * 70)
    display(results_df)

    print("\nDetailed comparison:")
    print(f"{'Metric':<20} {'Raw':>12} {'Denoised':>12} {'Change':>12}")
    print("-" * 60)

    for metric in ["ARI", "NMI", "silhouette"]:
        raw_val = results["Raw observed counts"][metric]
        den_val = results["Step4 denoised"][metric]
        change = den_val - raw_val
        symbol = "✓" if change > 0 else "✗"
        print(f"{metric:<20} {raw_val:>12.4f} {den_val:>12.4f} {change:>+12.4f} {symbol}")

    print(
        f"{'n_clusters':<20} "
        f"{results['Raw observed counts']['n_clusters']:>12} "
        f"{results['Step4 denoised']['n_clusters']:>12} "
        f"{results['Step4 denoised']['n_clusters'] - results['Raw observed counts']['n_clusters']:>+12}"
    )

    # --------------------------------------------------------------------------
    # 11. Save result table
    # --------------------------------------------------------------------------
    results_df.to_csv("/content/clustering_quality_scanpy_leiden.csv", index=False)
    print("\nSaved: /content/clustering_quality_scanpy_leiden.csv")

    # --------------------------------------------------------------------------
    # 12. Plot
    # --------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    metric_names = ["ARI", "NMI", "silhouette"]
    x = np.arange(len(metric_names))
    width = 0.35

    raw_vals = [results["Raw observed counts"][m] for m in metric_names]
    den_vals = [results["Step4 denoised"][m] for m in metric_names]

    axes[0].bar(x - width / 2, raw_vals, width, label="Raw observed counts")
    axes[0].bar(x + width / 2, den_vals, width, label="Step4 denoised")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metric_names)
    axes[0].set_ylabel("Score")
    axes[0].set_title("Cell-type structure metrics")
    axes[0].legend()

    for i, (r, d) in enumerate(zip(raw_vals, den_vals)):
        change = d - r
        axes[0].text(
            i,
            max(r, d) + 0.02,
            f"{change:+.3f}",
            ha="center",
            fontsize=9
        )

    axes[1].bar(
        ["Raw observed counts", "Step4 denoised"],
        [
            results["Raw observed counts"]["n_clusters"],
            results["Step4 denoised"]["n_clusters"],
        ]
    )
    axes[1].set_ylabel("Number of Leiden clusters")
    axes[1].set_title("Leiden cluster count")

    plt.tight_layout()
    plt.savefig("/content/clustering_quality_scanpy_leiden.png", dpi=150)
    plt.show()

    return results, results_df, adata_raw, adata_den


# ==============================================================================
# TASK 3: DIFFERENTIAL EXPRESSION
# ==============================================================================

def compare_differential_expression(denoised_adata, celltype1, celltype2, top_n=20):
    """
    Compare DE genes between two cell types using raw vs denoised data.

    This part keeps your original logic, using:
        raw = denoised_adata.layers['raw']
        denoised = denoised_adata.X
    """

    ct_col = get_cell_type_column(denoised_adata)

    all_types = denoised_adata.obs[ct_col].astype(str).unique()

    ct1_match = [ct for ct in all_types if celltype1.lower() in ct.lower()]
    ct2_match = [ct for ct in all_types if celltype2.lower() in ct.lower()]

    if not ct1_match or not ct2_match:
        print(f"Could not find cell types matching {celltype1} and {celltype2}")
        print(f"Available: {list(all_types)}")
        return None

    mask1 = denoised_adata.obs[ct_col].astype(str).isin(ct1_match).values
    mask2 = denoised_adata.obs[ct_col].astype(str).isin(ct2_match).values

    print(f"Comparing {ct1_match} (n={mask1.sum()}) vs {ct2_match} (n={mask2.sum()})")

    X_raw = ensure_dense(denoised_adata.layers["raw"])
    X_den = ensure_dense(denoised_adata.X)
    genes = denoised_adata.var_names

    de_results = []

    for i, gene in enumerate(genes):
        raw1, raw2 = X_raw[mask1, i], X_raw[mask2, i]
        den1, den2 = X_den[mask1, i], X_den[mask2, i]

        stat_raw, pval_raw = mannwhitneyu(raw1, raw2, alternative="two-sided")
        stat_den, pval_den = mannwhitneyu(den1, den2, alternative="two-sided")

        fc_raw = np.log2((raw1.mean() + 0.1) / (raw2.mean() + 0.1))
        fc_den = np.log2((den1.mean() + 0.1) / (den2.mean() + 0.1))

        de_results.append({
            "gene": gene,
            "pval_raw": pval_raw,
            "pval_den": pval_den,
            "fc_raw": fc_raw,
            "fc_den": fc_den,
            "more_sig_den": pval_den < pval_raw,
        })

    de_df = pd.DataFrame(de_results)

    alpha = 0.05 / len(genes)

    n_sig_raw = int((de_df["pval_raw"] < alpha).sum())
    n_sig_den = int((de_df["pval_den"] < alpha).sum())

    print(f"\nSignificant DE genes, Bonferroni alpha={alpha:.2e}:")
    print(f"  Raw: {n_sig_raw}")
    print(f"  Denoised: {n_sig_den}")
    print(f"  Improvement: {n_sig_den - n_sig_raw:+d}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, (pval_col, fc_col, title) in zip(
        axes,
        [
            ("pval_raw", "fc_raw", "Raw"),
            ("pval_den", "fc_den", "Denoised"),
        ]
    ):
        sig = de_df[pval_col] < alpha

        ax.scatter(
            de_df.loc[~sig, fc_col],
            -np.log10(de_df.loc[~sig, pval_col] + 1e-300),
            alpha=0.3,
            s=5,
            c="gray",
            label="Not significant"
        )

        ax.scatter(
            de_df.loc[sig, fc_col],
            -np.log10(de_df.loc[sig, pval_col] + 1e-300),
            alpha=0.7,
            s=10,
            c="red",
            label=f"Significant, n={sig.sum()}"
        )

        ax.axhline(-np.log10(alpha), color="black", linestyle="--", alpha=0.5)
        ax.axvline(0, color="black", linestyle="-", alpha=0.3)

        ax.set_xlabel("Log2 Fold Change")
        ax.set_ylabel("-Log10 p-value")
        ax.set_title(f"{title}: {ct1_match[0][:10]} vs {ct2_match[0][:10]}")
        ax.legend()

    plt.tight_layout()
    plt.savefig("/content/de_comparison.png", dpi=150)
    plt.show()

    de_df["neg_log_p_den"] = -np.log10(de_df["pval_den"] + 1e-300)

    top_genes = de_df.nlargest(top_n, "neg_log_p_den")[
        ["gene", "fc_den", "pval_den", "pval_raw"]
    ]

    print(f"\nTop {top_n} DE genes, denoised:")
    display(top_genes)

    return de_df


# ==============================================================================
# TASK 4: SVG DETECTION
# ==============================================================================

def compare_svg_detection(denoised_adata, method="moran"):
    """
    Compare spatially variable genes detected in raw vs denoised data.
    Uses a simple Moran's I style analysis.
    """

    ct_col = get_cell_type_column(denoised_adata)

    n_sample = min(5000, denoised_adata.n_obs)

    rng = np.random.default_rng(42)
    idx = rng.choice(denoised_adata.n_obs, n_sample, replace=False)

    adata_sub = denoised_adata[idx].copy()

    x_col = "x_centroid" if "x_centroid" in adata_sub.obs.columns else "x"
    y_col = "y_centroid" if "y_centroid" in adata_sub.obs.columns else "y"

    if x_col not in adata_sub.obs.columns or y_col not in adata_sub.obs.columns:
        print("No spatial coordinate columns found. Skipping SVG detection.")
        return None

    adata_sub.obsm["spatial"] = np.column_stack([
        adata_sub.obs[x_col].values,
        adata_sub.obs[y_col].values,
    ])

    adata_raw = adata_sub.copy()
    adata_raw.X = ensure_dense(adata_sub.layers["raw"]).astype(np.float32)

    adata_den = adata_sub.copy()
    adata_den.X = ensure_dense(adata_sub.X).astype(np.float32)

    sc.pp.normalize_total(adata_raw, target_sum=1e4)
    sc.pp.log1p(adata_raw)

    sc.pp.normalize_total(adata_den, target_sum=1e4)
    sc.pp.log1p(adata_den)

    sc.pp.neighbors(adata_raw, use_rep="spatial", n_neighbors=15)
    sc.pp.neighbors(adata_den, use_rep="spatial", n_neighbors=15)

    def compute_morans_i(adata):
        W = adata.obsp["connectivities"]

        row_sums = np.asarray(W.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        W = W.multiply(1 / row_sums[:, None])

        X = ensure_dense(adata.X)

        n = X.shape[0]
        morans = []

        for j in range(X.shape[1]):
            x = X[:, j]
            x_mean = x.mean()
            x_dev = x - x_mean

            numerator = (W @ x_dev) * x_dev
            denominator = (x_dev ** 2).sum()

            if denominator > 0:
                I = n * numerator.sum() / (W.sum() * denominator)
            else:
                I = 0.0

            morans.append(I)

        return np.array(morans)

    print("Computing Moran's I for raw data...")
    morans_raw = compute_morans_i(adata_raw)

    print("Computing Moran's I for denoised data...")
    morans_den = compute_morans_i(adata_den)

    genes = adata_sub.var_names

    svg_df = pd.DataFrame({
        "gene": genes,
        "moran_raw": morans_raw,
        "moran_den": morans_den,
    })

    threshold = 0.1

    svg_raw = set(svg_df[svg_df["moran_raw"] > threshold]["gene"])
    svg_den = set(svg_df[svg_df["moran_den"] > threshold]["gene"])

    print(f"\nSVGs detected, Moran's I > {threshold}:")
    print(f"  Raw: {len(svg_raw)}")
    print(f"  Denoised: {len(svg_den)}")
    print(f"  Overlap: {len(svg_raw & svg_den)}")
    print(f"  New in denoised: {len(svg_den - svg_raw)}")
    print(f"  Lost in denoised: {len(svg_raw - svg_den)}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(svg_df["moran_raw"], svg_df["moran_den"], alpha=0.5, s=10)

    max_val = max(svg_df["moran_raw"].max(), svg_df["moran_den"].max())
    axes[0].plot([0, max_val], [0, max_val], "r--")

    axes[0].axhline(threshold, color="gray", linestyle="--", alpha=0.5)
    axes[0].axvline(threshold, color="gray", linestyle="--", alpha=0.5)

    axes[0].set_xlabel("Moran's I, Raw")
    axes[0].set_ylabel("Moran's I, Denoised")
    axes[0].set_title("Spatial variability: Raw vs Denoised")

    categories = ["Only Raw", "Both", "Only Denoised"]
    values = [
        len(svg_raw - svg_den),
        len(svg_raw & svg_den),
        len(svg_den - svg_raw),
    ]

    axes[1].bar(categories, values)
    axes[1].set_ylabel("Number of SVGs")
    axes[1].set_title(f"SVG comparison, threshold={threshold}")

    for i, v in enumerate(values):
        axes[1].text(i, v + 1, str(v), ha="center")

    plt.tight_layout()
    plt.savefig("/content/svg_comparison.png", dpi=150)
    plt.show()

    return svg_df


# ==============================================================================
# RUN ALL VALIDATIONS
# ==============================================================================

def run_all_validations(denoised_adata):
    """
    Run all downstream validation tasks.
    """

    print("=" * 70)
    print("DOWNSTREAM VALIDATION FOR CELL-LEVEL DENOISING")
    print("=" * 70)

    print("\n" + "=" * 70)
    print("TASK 1: MARKER GENE VALIDATION")
    print("=" * 70)

    marker_results = validate_markers(denoised_adata)

    print("\n" + "=" * 70)
    print("TASK 2: CLUSTERING QUALITY")
    print("=" * 70)

    clustering_results, clustering_df, adata_raw_clustered, adata_den_clustered = (
        evaluate_clustering_quality(
            denoised_adata,
            raw_matrix=X_raw_counts,
            sample_size=50000,
            random_state=42,
            n_pcs=30,
            leiden_resolution=0.5,
        )
    )

    print("\n" + "=" * 70)
    print("TASK 3: DIFFERENTIAL EXPRESSION")
    print("=" * 70)

    de_results = compare_differential_expression(
        denoised_adata,
        "tumor",
        "t_cell"
    )

    print("\n" + "=" * 70)
    print("TASK 4: SVG DETECTION")
    print("=" * 70)

    svg_results = compare_svg_detection(denoised_adata)

    print("\n" + "=" * 70)
    print("VALIDATION COMPLETE")
    print("=" * 70)

    return {
        "markers": marker_results,
        "clustering": clustering_results,
        "clustering_df": clustering_df,
        "adata_raw_clustered": adata_raw_clustered,
        "adata_den_clustered": adata_den_clustered,
        "de": de_results,
        "svg": svg_results,
    }


# ==============================================================================
# RUN
# ==============================================================================

results = run_all_validations(denoised_adata)

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

# ============================================================
# UMAP: RAW VS DENOISED DATA
# Coherent with downstream validation:
#   raw      = X_raw_counts / denoised_adata.layers['raw']
#   denoised = denoised_adata.X
#   labels   = cell_type
# ============================================================

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

RANDOM_STATE = 42
N_PCS = 30
N_NEIGHBORS = 15

# ------------------------------------------------------------
# 1. Ensure cell_type exists
# ------------------------------------------------------------

if "cell_type" not in denoised_adata.obs.columns:
    if "cell_type" in annotated_spatial_adata.obs.columns:
        denoised_adata.obs["cell_type"] = (
            annotated_spatial_adata.obs.loc[denoised_adata.obs_names, "cell_type"]
            .astype(str)
            .values
        )
        print("Copied cell_type from annotated_spatial_adata to denoised_adata.")
    elif "xenium_labels" in globals() and "Barcode" in xenium_labels.columns and "Assigned_Xenium_Cell_Type" in xenium_labels.columns:
        label_map = xenium_labels.set_index(xenium_labels["Barcode"].astype(str))["Assigned_Xenium_Cell_Type"]
        denoised_adata.obs["cell_type"] = denoised_adata.obs_names.astype(str).map(label_map)
        print("Mapped cell_type from xenium_labels.")
    else:
        raise KeyError(
            "cell_type column not found in denoised_adata or annotated_spatial_adata, "
            "and could not map from xenium_labels."
        )

# Check missing labels
missing_labels = denoised_adata.obs["cell_type"].isna().sum()
print(f"Missing cell_type labels: {missing_labels:,}")

if missing_labels > 0:
    raise ValueError(
        f"{missing_labels:,} cells have missing cell_type labels. "
        "Fix labels before plotting UMAP."
    )

# ------------------------------------------------------------
# 2. Ensure raw layer matches X_raw_counts
# ------------------------------------------------------------

if "X_raw_counts" in globals():
    if X_raw_counts.shape != denoised_adata.X.shape:
        raise ValueError(
            f"Shape mismatch: X_raw_counts {X_raw_counts.shape} vs "
            f"denoised_adata.X {denoised_adata.X.shape}"
        )

    denoised_adata.layers["raw"] = X_raw_counts.astype(np.float32).copy()
    print("Set denoised_adata.layers['raw'] = X_raw_counts.")
else:
    if "raw" not in denoised_adata.layers:
        raise KeyError("X_raw_counts not found and denoised_adata.layers['raw'] is missing.")
    print("Using existing denoised_adata.layers['raw'].")

print(f"Raw sum: {ensure_dense(denoised_adata.layers['raw']).sum(dtype=np.float64):,.0f}")
print(f"Denoised sum: {ensure_dense(denoised_adata.X).sum(dtype=np.float64):,.2f}")

# ------------------------------------------------------------
# 3. Create temporary AnnData objects
# ------------------------------------------------------------

adata_raw_for_umap = denoised_adata.copy()
adata_raw_for_umap.X = ensure_dense(denoised_adata.layers["raw"]).astype(np.float32).copy()

adata_den_for_umap = denoised_adata.copy()
adata_den_for_umap.X = ensure_dense(denoised_adata.X).astype(np.float32).copy()

# ------------------------------------------------------------
# 4. Process raw data
# ------------------------------------------------------------

print("Processing raw data for UMAP...")

sc.pp.normalize_total(adata_raw_for_umap, target_sum=1e4)
sc.pp.log1p(adata_raw_for_umap)
sc.pp.pca(adata_raw_for_umap, n_comps=N_PCS, random_state=RANDOM_STATE)
sc.pp.neighbors(
    adata_raw_for_umap,
    n_neighbors=N_NEIGHBORS,
    n_pcs=N_PCS,
    random_state=RANDOM_STATE
)
sc.tl.umap(adata_raw_for_umap, random_state=RANDOM_STATE)

# ------------------------------------------------------------
# 5. Process denoised data
# ------------------------------------------------------------

print("Processing denoised data for UMAP...")

sc.pp.normalize_total(adata_den_for_umap, target_sum=1e4)
sc.pp.log1p(adata_den_for_umap)
sc.pp.pca(adata_den_for_umap, n_comps=N_PCS, random_state=RANDOM_STATE)
sc.pp.neighbors(
    adata_den_for_umap,
    n_neighbors=N_NEIGHBORS,
    n_pcs=N_PCS,
    random_state=RANDOM_STATE
)
sc.tl.umap(adata_den_for_umap, random_state=RANDOM_STATE)

# ------------------------------------------------------------
# 6. Plot
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

sc.pl.umap(
    adata_raw_for_umap,
    color="cell_type",
    title="UMAP: Raw Data",
    ax=axes[0],
    show=False,
    legend_loc="on data"
)
axes[0].set_aspect("equal", adjustable="box")

sc.pl.umap(
    adata_den_for_umap,
    color="cell_type",
    title="UMAP: Denoised Data",
    ax=axes[1],
    show=False,
    legend_loc="on data"
)
axes[1].set_aspect("equal", adjustable="box")

plt.suptitle("UMAP Visualization of Cell Types Before and After Denoising", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("/content/umap_raw_vs_denoised.png", dpi=300)
plt.show()

print("UMAP plots saved to /content/umap_raw_vs_denoised.png")

===========DONE===========

In [ ]:
# ==============================================================================
# CELL 1: Install dependencies
# ==============================================================================

# !pip install spatialde --quiet --force-reinstall
# Note: SpatialDE can be slow to install. If issues, skip to SPARK method.

In [ ]:
"""
================================================================================
COLAB CELLS - SVG ANALYSIS WITH SpatialDE AND SPARK
================================================================================
Make sure denoised_adata has:
    - .X = denoised expression
    - .layers['raw'] = raw expression
    - .obs with x_centroid, y_centroid

================================================================================
"""

# ==============================================================================
# CELL 1: Install dependencies
# ==============================================================================

!pip install spatialde --quiet
# Note: SpatialDE can be slow to install. If issues, skip to SPARK method.

print("Dependencies ready")


# ==============================================================================
# CELL 2: Helper functions
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, norm
import warnings
warnings.filterwarnings('ignore')

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return X


# ==============================================================================
# CELL 3: SpatialDE Analysis
# ==============================================================================

def run_spatialde_comparison(denoised_adata, n_samples=5000, fdr_threshold=0.05):
    """
    Run SpatialDE on raw vs denoised data.
    """
    try:
        import SpatialDE
    except ImportError:
        print("SpatialDE not installed. Run: !pip install spatialde")
        print("Skipping to SPARK method...")
        return None

    print("=" * 60)
    print("SpatialDE ANALYSIS")
    print("=" * 60)

    # Subsample
    if denoised_adata.n_obs > n_samples:
        idx = np.random.choice(denoised_adata.n_obs, n_samples, replace=False)
        adata_sub = denoised_adata[idx].copy()
        print(f"Subsampled to {n_samples} cells")
    else:
        adata_sub = denoised_adata.copy()

    # Get coordinates
    x_col = 'x_centroid' if 'x_centroid' in adata_sub.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in adata_sub.obs else 'y'

    coords = pd.DataFrame({
        'x': adata_sub.obs[x_col].values,
        'y': adata_sub.obs[y_col].values
    }, index=adata_sub.obs_names)

    results = {}

    for layer_name, layer in [('raw', 'raw'), ('denoised', 'X')]:
        print(f"\nProcessing {layer_name}...")

        if layer == 'X':
            expr = ensure_dense(adata_sub.X)
        else:
            expr = ensure_dense(adata_sub.layers[layer])

        counts = pd.DataFrame(expr, index=adata_sub.obs_names, columns=adata_sub.var_names)

        # Normalize
        norm_expr = SpatialDE.NaiveDE.stabilize(counts.T).T

        # Run SpatialDE
        print(f"  Running SpatialDE...")
        res = SpatialDE.run(coords, norm_expr)
        res = res.sort_values('pval')

        results[layer_name] = res
        print(f"  Found {(res['qval'] < fdr_threshold).sum()} significant SVGs")

    # Compare
    svg_raw = set(results['raw'][results['raw']['qval'] < fdr_threshold]['g'])
    svg_den = set(results['denoised'][results['denoised']['qval'] < fdr_threshold]['g'])

    print(f"\n" + "=" * 60)
    print("RESULTS")
    print("=" * 60)
    print(f"Raw SVGs: {len(svg_raw)}")
    print(f"Denoised SVGs: {len(svg_den)}")
    print(f"Overlap: {len(svg_raw & svg_den)}")
    print(f"NEW in denoised: {len(svg_den - svg_raw)}")

    # Plot
    merged = results['raw'][['g', 'pval', 'qval']].merge(
        results['denoised'][['g', 'pval', 'qval']],
        on='g', suffixes=('_raw', '_den')
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    ax.scatter(-np.log10(merged['pval_raw']), -np.log10(merged['pval_den']), alpha=0.5, s=10)
    ax.plot([0, 50], [0, 50], 'r--')
    ax.set_xlabel('-log10(p) Raw')
    ax.set_ylabel('-log10(p) Denoised')
    ax.set_title('SpatialDE P-values')

    ax = axes[1]
    categories = ['Only Raw', 'Both', 'Only Denoised']
    values = [len(svg_raw - svg_den), len(svg_raw & svg_den), len(svg_den - svg_raw)]
    ax.bar(categories, values, color=['lightblue', 'purple', 'coral'])
    ax.set_ylabel('Number of SVGs')
    ax.set_title(f'SVG Comparison (FDR < {fdr_threshold})')

    plt.tight_layout()
    plt.savefig('/content/spatialde_results.png', dpi=150)
    plt.show()

    return {'raw': results['raw'], 'denoised': results['denoised'],
            'svg_raw': svg_raw, 'svg_den': svg_den}


# ==============================================================================
# CELL 4: SPARK-like Analysis (doesn't require R)
# ==============================================================================

def run_spark_comparison(denoised_adata, n_samples=5000, fdr_threshold=0.05):
    """
    Run SPARK-like spatial analysis (kernel-based spatial autocorrelation).
    This is a Python implementation that doesn't require R.
    """
    print("=" * 60)
    print("SPARK-like ANALYSIS")
    print("=" * 60)

    # Subsample
    if denoised_adata.n_obs > n_samples:
        idx = np.random.choice(denoised_adata.n_obs, n_samples, replace=False)
        adata_sub = denoised_adata[idx].copy()
        print(f"Subsampled to {n_samples} cells")
    else:
        adata_sub = denoised_adata.copy()

    # Get coordinates
    x_col = 'x_centroid' if 'x_centroid' in adata_sub.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in adata_sub.obs else 'y'
    coords = np.column_stack([adata_sub.obs[x_col].values, adata_sub.obs[y_col].values])

    # Compute spatial kernel
    print("Computing spatial kernel...")
    D = squareform(pdist(coords))
    bandwidth = np.median(D[D > 0])
    K = np.exp(-D**2 / (2 * bandwidth**2))
    np.fill_diagonal(K, 0)
    K = K / (K.sum(axis=1, keepdims=True) + 1e-8)

    results = {}

    for layer_name, layer in [('raw', 'raw'), ('denoised', 'X')]:
        print(f"\nProcessing {layer_name}...")

        if layer == 'X':
            expr = ensure_dense(adata_sub.X)
        else:
            expr = ensure_dense(adata_sub.layers[layer])

        # Normalize
        lib_sizes = expr.sum(axis=1, keepdims=True)
        expr_norm = np.log1p(expr / (lib_sizes + 1e-8) * 1e4)

        # Compute spatial autocorrelation per gene
        result_list = []
        n_genes = expr_norm.shape[1]

        for j in range(n_genes):
            gene = adata_sub.var_names[j]
            y = expr_norm[:, j]
            y_c = y - y.mean()

            if y_c.std() > 0:
                Wy = K @ y_c
                spatial_corr, _ = pearsonr(y_c, Wy)

                # Z-score for p-value
                n = len(y)
                z = spatial_corr * np.sqrt(n - 2) / np.sqrt(1 - spatial_corr**2 + 1e-8)
                pval = 2 * (1 - norm.cdf(abs(z)))
            else:
                spatial_corr = 0
                pval = 1.0

            result_list.append({'gene': gene, 'spatial_corr': spatial_corr, 'pval': pval})

        res = pd.DataFrame(result_list)

        # FDR correction (BH)
        pvals = res['pval'].values
        n = len(pvals)
        sorted_idx = np.argsort(pvals)
        qvals = np.ones(n)
        for i, idx in enumerate(sorted_idx):
            qvals[idx] = min(1, pvals[idx] * n / (i + 1))
        # Make monotonic
        for i in range(n-2, -1, -1):
            qvals[sorted_idx[i]] = min(qvals[sorted_idx[i]], qvals[sorted_idx[i+1]])
        res['qval'] = qvals

        res = res.sort_values('pval')
        results[layer_name] = res
        print(f"  Found {(res['qval'] < fdr_threshold).sum()} significant SVGs")

    # Compare
    svg_raw = set(results['raw'][results['raw']['qval'] < fdr_threshold]['gene'])
    svg_den = set(results['denoised'][results['denoised']['qval'] < fdr_threshold]['gene'])

    print(f"\n" + "=" * 60)
    print("RESULTS")
    print("=" * 60)
    print(f"Raw SVGs: {len(svg_raw)}")
    print(f"Denoised SVGs: {len(svg_den)}")
    print(f"Overlap: {len(svg_raw & svg_den)}")
    print(f"NEW in denoised: {len(svg_den - svg_raw)}")
    print(f"LOST in denoised: {len(svg_raw - svg_den)}")

    # New SVGs
    if svg_den - svg_raw:
        print(f"\nTop NEW SVGs (found only after denoising):")
        new_svgs = results['denoised'][results['denoised']['gene'].isin(svg_den - svg_raw)]
        print(new_svgs.head(10)[['gene', 'spatial_corr', 'qval']].to_string(index=False))

    # Plot
    merged = results['raw'][['gene', 'pval', 'spatial_corr']].merge(
        results['denoised'][['gene', 'pval', 'spatial_corr']],
        on='gene', suffixes=('_raw', '_den')
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # P-value comparison
    ax = axes[0]
    ax.scatter(-np.log10(merged['pval_raw'] + 1e-300),
               -np.log10(merged['pval_den'] + 1e-300), alpha=0.5, s=10)
    max_val = max(-np.log10(merged['pval_raw'] + 1e-300).max(),
                  -np.log10(merged['pval_den'] + 1e-300).max())
    ax.plot([0, max_val], [0, max_val], 'r--')
    ax.set_xlabel('-log10(p) Raw')
    ax.set_ylabel('-log10(p) Denoised')
    ax.set_title('SPARK P-values')

    # Spatial correlation comparison
    ax = axes[1]
    ax.scatter(merged['spatial_corr_raw'], merged['spatial_corr_den'], alpha=0.5, s=10)
    ax.plot([-0.5, 1], [-0.5, 1], 'r--')
    ax.set_xlabel('Spatial Corr (Raw)')
    ax.set_ylabel('Spatial Corr (Denoised)')
    ax.set_title('Spatial Autocorrelation')

    # Venn
    ax = axes[2]
    categories = ['Only Raw', 'Both', 'Only Denoised']
    values = [len(svg_raw - svg_den), len(svg_raw & svg_den), len(svg_den - svg_raw)]
    bars = ax.bar(categories, values, color=['lightblue', 'purple', 'coral'])
    ax.set_ylabel('Number of SVGs')
    ax.set_title(f'SVG Comparison (FDR < {fdr_threshold})')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', fontsize=12)

    plt.tight_layout()
    plt.savefig('/content/spark_results.png', dpi=150)
    plt.show()

    return {'raw': results['raw'], 'denoised': results['denoised'],
            'svg_raw': svg_raw, 'svg_den': svg_den}


# ==============================================================================
# CELL 5: Run the analysis
# ==============================================================================

# Make sure denoised_adata is ready
print(f"Data: {denoised_adata.n_obs} cells × {denoised_adata.n_vars} genes")
print(f"Layers: {list(denoised_adata.layers.keys())}")

# Run SPARK-like analysis (no extra dependencies)
spark_results = run_spark_comparison(denoised_adata, n_samples=5000, fdr_threshold=0.05)

# Optionally run SpatialDE (requires installation)
spatialde_results = run_spatialde_comparison(denoised_adata, n_samples=5000, fdr_threshold=0.05)


# ==============================================================================
# CELL 6: Visualize top SVGs spatially
# ==============================================================================

def plot_top_svgs(denoised_adata, spark_results, n_genes=6):
    """Plot spatial expression of top SVGs from denoised data."""

    top_svgs = spark_results['denoised'].head(n_genes)['gene'].tolist()

    x_col = 'x_centroid' if 'x_centroid' in denoised_adata.obs else 'x'
    y_col = 'y_centroid' if 'y_centroid' in denoised_adata.obs else 'y'

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for i, gene in enumerate(top_svgs):
        ax = axes[i]

        gene_idx = denoised_adata.var_names.tolist().index(gene)
        expr = denoised_adata.X[:, gene_idx]

        # Subsample for plotting
        if len(expr) > 10000:
            idx = np.random.choice(len(expr), 10000, replace=False)
        else:
            idx = np.arange(len(expr))

        x = denoised_adata.obs[x_col].values[idx]
        y = denoised_adata.obs[y_col].values[idx]
        c = expr[idx]

        scatter = ax.scatter(x, y, c=c, s=1, alpha=0.7, cmap='viridis')
        ax.set_title(f'{gene}')
        ax.set_aspect('equal')
        ax.axis('off')
        plt.colorbar(scatter, ax=ax, shrink=0.5)

    plt.suptitle('Top Spatially Variable Genes (Denoised)', fontsize=14)
    plt.tight_layout()
    plt.savefig('/content/top_svgs_spatial.png', dpi=150)
    plt.show()

# Plot top SVGs
plot_top_svgs(denoised_adata, spark_results)


# ==============================================================================
# CELL 7: Summary for thesis
# ==============================================================================

print("=" * 60)
print("SVG ANALYSIS SUMMARY FOR THESIS")
print("=" * 60)

svg_raw = spark_results['svg_raw']
svg_den = spark_results['svg_den']

print(f"""
SPATIALLY VARIABLE GENE DETECTION

Method: SPARK-like kernel-based spatial autocorrelation
Cells analyzed: {min(5000, denoised_adata.n_obs)}
FDR threshold: 0.05

RESULTS:
  SVGs in raw data: {len(svg_raw)}
  SVGs in denoised data: {len(svg_den)}

  Overlap: {len(svg_raw & svg_den)} genes
  Lost after denoising: {len(svg_raw - svg_den)} genes
  NEW after denoising: {len(svg_den - svg_raw)} genes

INTERPRETATION:
  - Denoising {'recovered' if len(svg_den) > len(svg_raw) else 'maintained'} {abs(len(svg_den) - len(svg_raw))} SVGs
  - {len(svg_den - svg_raw)} genes became detectable only after denoising
  - This suggests {'improved' if len(svg_den) >= len(svg_raw) else 'maintained'} spatial signal detection
""")

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

def ensure_dense(X):
    """Convert sparse to dense if needed."""
    if sparse.issparse(X):
        return X.toarray()
    return X

n_samples = 5000 # Keep this consistent with the run_spatialde_comparison function

# Subsample the denoised_adata if it has more cells than n_samples
if denoised_adata.n_obs > n_samples:
    idx = np.random.choice(denoised_adata.n_obs, n_samples, replace=False)
    adata_sub = denoised_adata[idx].copy()
    print(f"Subsampled denoised_adata to {n_samples} cells for export.")
else:
    adata_sub = denoised_adata.copy()
    print(f"Using full denoised_adata ({denoised_adata.n_obs} cells) for export.")

# 1. Extract and save spatial coordinates
x_col = 'x_centroid' if 'x_centroid' in adata_sub.obs else 'x'
y_col = 'y_centroid' if 'y_centroid' in adata_sub.obs else 'y'

coords_df = adata_sub.obs[[x_col, y_col]].copy()
coords_df.columns = ['x', 'y'] # Rename for SpatialDE compatibility
coords_df.to_csv('coordinates.csv', index=True) # index is cell_id
print(f"Saved coordinates to coordinates.csv (shape: {coords_df.shape})")
display(coords_df.head())

# 2. Extract and save raw expression data
raw_expr = ensure_dense(adata_sub.layers['raw'])
raw_counts_df = pd.DataFrame(raw_expr, index=adata_sub.obs_names, columns=adata_sub.var_names)
raw_counts_df.to_csv('raw_expression.csv', index=True) # index is cell_id, columns are gene_names
print(f"Saved raw expression to raw_expression.csv (shape: {raw_counts_df.shape})")
display(raw_counts_df.head())

# 3. Extract and save denoised expression data
denoised_expr = ensure_dense(adata_sub.X)
denoised_counts_df = pd.DataFrame(denoised_expr, index=adata_sub.obs_names, columns=adata_sub.var_names)
denoised_counts_df.to_csv('denoised_expression.csv', index=True)
print(f"Saved denoised expression to denoised_expression.csv (shape: {denoised_counts_df.shape})")
display(denoised_counts_df.head())

# 4. Save gene names
gene_names_df = pd.DataFrame({'gene_name': adata_sub.var_names})
gene_names_df.to_csv('gene_names.txt', index=False, header=False)
print(f"Saved gene names to gene_names.txt ({len(gene_names_df)} genes)")


In [ ]:
# ==============================================================================
# CELL: EXPORT
# ==============================================================================
# This saves everything Step 5 needs into a single folder.
# After running, download the folder (or mount Google Drive to save directly).

import numpy as np
import pandas as pd
from scipy import sparse
import os, json, time

def ensure_dense(X):
    if sparse.issparse(X):
        return X.toarray()
    return np.asarray(X)

EXPORT_DIR = "/content/step4_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

print("=" * 60)
print("EXPORTING FILES FOR STEP 5")
print("=" * 60)

# ---------------------------------------------------------------
# FILE 1: molecules.parquet (already exists, just verify + copy)
# ---------------------------------------------------------------
import shutil
src_mol = "/content/molecules.parquet"
dst_mol = os.path.join(EXPORT_DIR, "molecules.parquet")
if os.path.exists(src_mol):
    shutil.copy2(src_mol, dst_mol)
    mol_check = pd.read_parquet(dst_mol, columns=['cell_id']).shape[0]
    print(f"1. molecules.parquet: {mol_check:,} rows — COPIED")
else:
    print("ERROR: molecules.parquet not found at /content/")

# ---------------------------------------------------------------
# FILE 2: cell_data.npz (already exists, just verify + copy)
# ---------------------------------------------------------------
src_geom = "/content/cell_data.npz"
dst_geom = os.path.join(EXPORT_DIR, "cell_data.npz")
if os.path.exists(src_geom):
    shutil.copy2(src_geom, dst_geom)
    geom_check = np.load(dst_geom, allow_pickle=False)
    print(f"2. cell_data.npz: {len(geom_check['cell_ids']):,} cells — COPIED")
else:
    print("ERROR: cell_data.npz not found at /content/")

# ---------------------------------------------------------------
# FILE 3: denoised_adata.h5ad (THE MAIN STEP 4 OUTPUT)
# ---------------------------------------------------------------
# This contains:
#   .X          = X_denoised (118752 x 313)
#   .layers['raw']         = original raw counts
#   .layers['uncertainty'] = per-(cell, gene) uncertainty
#   .obs        = cell metadata including cell_type, was_corrected_count
#   .var_names  = shared gene names (313)
#   .obs_names  = cell IDs

if "X_raw_counts" in globals():
    if X_raw_counts.shape != denoised_adata.X.shape:
        raise ValueError(
            f"X_raw_counts shape {X_raw_counts.shape} does not match "
            f"denoised_adata.X shape {denoised_adata.X.shape}"
        )
    denoised_adata.layers["raw"] = X_raw_counts.astype(np.float32).copy()
    print("Updated denoised_adata.layers['raw'] with clean X_raw_counts before export.")

dst_adata = os.path.join(EXPORT_DIR, "denoised_adata.h5ad")
denoised_adata.write(dst_adata)
print(f"3. denoised_adata.h5ad: {denoised_adata.shape} — SAVED")
print(f"   .X = denoised counts")
print(f"   .layers = {list(denoised_adata.layers.keys())}")
print(f"   .obs columns = {list(denoised_adata.obs.columns)}")

# ---------------------------------------------------------------
# FILE 4: was_corrected.npy (boolean mask from Step 4)
# ---------------------------------------------------------------
dst_wc = os.path.join(EXPORT_DIR, "was_corrected.npy")
np.save(dst_wc, metadata['was_corrected'])
print(f"4. was_corrected.npy: {metadata['was_corrected'].shape} — SAVED")

# ---------------------------------------------------------------
# FILE 5: step4_config.json (parameters + metadata for reproducibility)
# ---------------------------------------------------------------
config = {
    'shared_genes': list(metadata['shared_genes']),
    'n_cells': int(denoised_adata.shape[0]),
    'n_genes': int(denoised_adata.shape[1]),
    'cell_type_column': 'cell_type' if 'cell_type' in denoised_adata.obs.columns
                        else 'Assigned_Xenium_Cell_Type',
    'cell_types': sorted(denoised_adata.obs[
        'cell_type' if 'cell_type' in denoised_adata.obs.columns
        else 'Assigned_Xenium_Cell_Type'
    ].unique().tolist()),
    'step4_params': {
        'spatial_weight': 0.5,
        'reference_weight': 0.7,
        'spatial_radius': 100.0,
        'dropout_p_threshold': 0.35,
        'shrinkage': 0.87,
        'calibrate': False,
    },
    'correction_stats': {
        'total_pairs': int(metadata['was_corrected'].size),
        'corrected_pairs': int(metadata['was_corrected'].sum()),
        'correction_rate': float(metadata['was_corrected'].sum() / metadata['was_corrected'].size),
    }
}
dst_config = os.path.join(EXPORT_DIR, "step4_config.json")
with open(dst_config, 'w') as f:
    json.dump(config, f, indent=2)
print(f"5. step4_config.json — SAVED")

# ---------------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------------
print(f"\n{'=' * 60}")
print(f"ALL FILES SAVED TO: {EXPORT_DIR}/")
print(f"{'=' * 60}")
for fname in sorted(os.listdir(EXPORT_DIR)):
    fpath = os.path.join(EXPORT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {fname:35s} {size_mb:8.1f} MB")
total_mb = sum(os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1e6
               for f in os.listdir(EXPORT_DIR))
print(f"  {'TOTAL':35s} {total_mb:8.1f} MB")

print(f"\nNEXT STEPS:")
print(f"  Option A: Download the folder from Colab file browser (left sidebar)")
print(f"  Option B: Run the Google Drive mount cell below to save permanently")


# ==============================================================================
# OPTIONAL: Save to Google Drive (run this after the export cell above)
# ==============================================================================

# Uncomment and run this cell to copy exports to your Google Drive:
from google.colab import drive
drive.mount('/content/drive')
import shutil
drive_dest = '/content/drive/MyDrive/diffusion/step4_exports'
os.makedirs(drive_dest, exist_ok=True)
for fname in os.listdir(EXPORT_DIR):
     shutil.copy2(os.path.join(EXPORT_DIR, fname), os.path.join(drive_dest, fname))
     print(f"Copied {fname} to Drive")
print(f"\nAll files saved to Google Drive: {drive_dest}")


## Parameters Used in the Notebook

This section lists the key parameters and their functions as defined and used throughout the analysis.

### 1. `denoise_v4_selective` Function Parameters

-   **`spatial_weight`**: Controls the influence of spatial proximity in the hybrid neighborhood definition. A value of `0.3` means 30% spatial, 70% expression.
-   **`reference_weight`**: Determines the influence of the scRNA-seq reference profiles when calculating the expected gene expression. A value of `0.3` means 30% reference, 70% local neighborhood.
-   **`spatial_radius`**: The radius (in microns) used by the `AdaptiveRadiusGraph` to define initial spatial neighbors. Here, it is set to `100.0`.
-   **`k_min`**: The minimum number of neighbors to consider for each cell in the `AdaptiveRadiusGraph`. Set to `5`.
-   **`k_max`**: The maximum number of neighbors to consider for each cell in the `AdaptiveRadiusGraph`. Set to `30`.
-   **`n_pca`**: The number of principal components to retain when performing PCA on the expression data for features. Set to `50`.
-   **`dropout_p_threshold`**: The p-value threshold for detecting dropouts. Values observed below this probability (given the expected expression) are considered potential dropouts. Here, it is set to `0.35`.
-   **`shrinkage`**: Controls how much a detected dropout value is replaced by its predicted value. A value of `0.87` means the corrected value is 87% the predicted mean and 13% the original observed value.
-   **`calibrate`**: A boolean flag (`True`/`False`) to enable or disable the uncertainty calibration step.
-   **`holdout_frac`**: The fraction of values to hold out for uncertainty calibration.

### 2. `UncertaintyCalibrator` Class Parameters

-   **`holdout_fraction`**: The proportion of data points to randomly mask for calibrating uncertainty (set to `0.1`).
-   **`target_coverage`**: The desired percentage of true values that should fall within ±1 standard deviation of the predicted value (set to `0.68` for 1-sigma coverage).

### 3. `AdaptiveRadiusGraph` Class Parameters

-   **`radius`**: The initial radius for querying neighbors (set to `100.0`).
-   **`k_min`**: Minimum number of neighbors to ensure robust neighborhood definitions (set to `5`).
-   **`k_max`**: Maximum number of neighbors to prevent overly large neighborhoods in dense regions (set to `30`).
-   **`adaptive_radius`**: A boolean flag (`True`/`False`) to enable or disable adaptive radius calculation based on local density.

### 4. `detect_dropouts_strict` Function Parameters

-   **`p_threshold`**: The probability threshold below which an observed count is considered a dropout (set to `0.01` internally in the function, but driven by `dropout_p_threshold` from `denoise_v4_selective`).

### 5. Global Parameters

-   **`DROPOUT_P`**: A global variable for the `dropout_p_threshold` used in `denoise_v4_selective`, set to `0.35`.
-   **`SHRINKAGE`**: A global variable for the `shrinkage` parameter used in `denoise_v4_selective`, set to `0.87`.
-   **`n_samples`**: The number of cells to subsample for computationally intensive SVG analyses (`SpatialDE`, `SPARK-like`). Set to `5000` to speed up computations.
-   **`fdr_threshold`**: The False Discovery Rate threshold for determining significant spatially variable genes (set to `0.05`).

### 6. `map_cell_types_improved` Function Parameters

-   **`synonyms`**: A dictionary of common cell type names and their synonyms, used for fuzzy matching between Xenium and scRNA-seq cell type labels.


*   **CV change (%)** (Coefficient of Variation change):
    *   **Meaning**: This metric quantifies the percentage change in the coefficient of variation (a measure of relative variability) of gene expression after denoising compared to the raw data. A negative value indicates a decrease in variability.
    *   **Good sign to decrease**: Yes. Denoising aims to reduce technical noise, which often manifests as high variability in gene expression. A decrease in CV (negative change) suggests that technical noise has been successfully reduced, leading to a clearer biological signal.

*   **Correction %**:
    *   **Meaning**: This represents the percentage of gene expression values that were identified as dropouts and subsequently corrected by the denoising algorithm. It indicates the extent of the denoising process.
    *   **Good sign to be informative, not strictly increase/decrease**: There isn't a universally 'good' direction for this metric to increase or decrease. It's primarily an informative metric. A high percentage might indicate significant technical noise in the raw data that was addressed, while a very low percentage might suggest minimal noise or a very conservative denoising approach. The 'ideal' percentage depends on the biological context and the noise level of the input data.

*   **Silhouette (score)**:
    *   **Meaning**: The Silhouette score measures how similar an object is to its own cluster compared to other clusters. It ranges from -1 to +1.
    *   **Good sign to increase**: Yes. A higher Silhouette score (closer to +1) indicates that cells within a cluster are tightly grouped and well-separated from other clusters, suggesting a better and more robust clustering structure.

*   **ARI (Adjusted Rand Index)**:
    *   **Meaning**: ARI measures the similarity between two data clusterings, adjusted for the probability of chance agreement. In this notebook, it compares the algorithm's clustering to 'true' or expert-annotated cell types.
    *   **Good sign to increase**: Yes. A higher ARI (closer to +1) indicates better agreement between the algorithm's clustering and the known, expert-defined cell types, implying that the denoised data yields more biologically accurate clusters.

*   **NMI (Normalized Mutual Information)**:
    *   **Meaning**: NMI quantifies the mutual dependence between two variables (here, the predicted clusters and the true cell types). Like ARI, it's a measure of clustering agreement.
    *   **Good sign to increase**: Yes. A higher NMI score (closer to +1) signifies greater mutual information between the predicted clusters and the true cell types, meaning the clustering captures more of the underlying biological structure present in the expert annotations.

*   **New SVG (Spatially Variable Genes)**:
    *   **Meaning**: This refers to the number of spatially variable genes (SVGs) that are detected as significant *only* after denoising, but not in the raw data. The notebook's context describes this as genes 'NEW after denoising'.
    *   **Good sign to increase**: Yes. The primary goal of identifying SVGs is to uncover biologically meaningful spatial patterns. If denoising reveals more SVGs that were previously obscured by noise, it indicates an improved ability to detect true biological spatial signals, enhancing the depth of analysis.

*   **DE (Differentially Expressed genes - significant count)**:
    *   **Meaning**: This refers to the number of genes found to be significantly differentially expressed between two cell types or conditions. The notebook compares the number of significant DE genes found in raw versus denoised data.
    *   **Good sign to increase**: Yes. Denoising aims to improve the signal-to-noise ratio, which should, in turn, increase the statistical power to detect true biological differences. Finding more significant DE genes in denoised data suggests that the algorithm has enhanced the ability to identify genuine biological distinctions between cell populations.

# ** NEXT STEP: MOLECULE-LEVEL IMPUTATION**